
# WI-03: Leakage-safe Feature Engineering

## 1. Objective

Build a reproducible, leakage-safe feature engineering pipeline for daily weather forecasting using the canonical `weather-daily-v1` dataset.

The engineered dataset will support forecasting:
- Daily precipitation (`PRECTOTCORR`, mm/day)
- Daily maximum temperature (`T2M_MAX`, °C)

Forecast horizons:
- T+1
- T+2
- T+3
- T+4
- T+5
- T+6
- T+7

## 2. Dataset

Canonical source:
`phase3b_weather_intelligence/data/processed/weather_daily_v1`

Dataset characteristics:
- Period: 2013-01-01 to 2024-12-31
- Grid cells: 2,520
- Observations: 11,045,160
- Spatial resolution: 0.5° latitude × 0.625° longitude
- Variables: `PRECTOTCORR`, `T2M_MAX`, `T2M_MIN`
- Temporal granularity: daily
- Canonical observation key: `(cell_id, date)`

## 3. Forecasting Targets

For each grid cell and forecast origin date t:

- Rainfall target at horizon h: `PRECTOTCORR(t+h)`
- Maximum temperature target at horizon h: `T2M_MAX(t+h)`

where h ranges from 1 to 7 days.

`T2M_MIN` will initially be used as an input feature, not a prediction target.

## 4. Feature Engineering Methodology

Candidate feature groups:

1. Historical lags:
   - 1, 2, 3, 7, 14 and 30 days

2. Rolling statistics:
   - 3, 7, 14 and 30 days
   - Means, standard deviations and selected sums

3. Calendar features:
   - Day of year
   - Month
   - Day of week
   - Cyclical calendar encodings

4. Spatial identifiers:
   - Grid-cell identity
   - Latitude and longitude

All lagged and rolling features must use information available on or before the forecast origin date.

Rolling calculations must exclude the current target day and all future observations.

## 5. Chronological Data Policy

Development period:
2013-01-01 to 2022-12-31

Final holdout:
2023-01-01 to 2024-12-31

The final holdout must not be used for:
- Feature selection
- Model selection
- Hyperparameter tuning
- Model comparison
- Training-period climatology estimation

Historical observations before a forecast origin may be used to construct its lagged features, including when the forecast origin falls near a split boundary.

The split is based on target dates, not merely feature-origin dates.

## 6. Important Constraints

- No APY, crop, yield or production variables.
- No future weather observations in input features.
- No random train-test splitting.
- No full-period climatology features.
- No interpolation that uses future observations.
- Preserve extreme weather observations.
- Do not silently remove valid grid-cell observations.
- Do not treat grid cells as administrative districts.

## 7. Expected Outputs

Store generated artifacts separately under:

`phase3b_weather_intelligence/data/processed/weather_features_v1/`

Supporting reports:

`phase3b_weather_intelligence/reports/weather_intelligence_v2/`

Expected deliverables:
- Feature specification
- Feature engineering validation report
- Leakage validation results
- Chronological target split summary
- Reproducible feature dataset

## 8. Handoff

WI-03 prepares the forecasting dataset for:
- WI-04: Forecasting Baselines
- WI-05: Rainfall Model Development
- WI-06: Temperature Model Development

No model training or final model selection is performed in this notebook.


In [1]:

# ============================================================
# WI-03 | CELL 2: ENVIRONMENT AND DATASET INITIALIZATION
# ============================================================

from pathlib import Path
import json
import platform

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds

from IPython.display import display

# ------------------------------------------------------------
# 1. Environment configuration
# ------------------------------------------------------------

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 42

START_DATE = pd.Timestamp("2013-01-01")
END_DATE = pd.Timestamp("2024-12-31")

DEVELOPMENT_START = pd.Timestamp("2013-01-01")
DEVELOPMENT_END = pd.Timestamp("2022-12-31")

FINAL_HOLDOUT_START = pd.Timestamp("2023-01-01")
FINAL_HOLDOUT_END = pd.Timestamp("2024-12-31")

FORECAST_HORIZONS = list(range(1, 8))

WEATHER_VARIABLES = [
    "PRECTOTCORR",
    "T2M_MAX",
    "T2M_MIN"
]

TARGET_VARIABLES = [
    "PRECTOTCORR",
    "T2M_MAX"
]

LAG_DAYS = [1, 2, 3, 7, 14, 30]
ROLLING_WINDOWS = [3, 7, 14, 30]

EXPECTED_GRID_CELLS = 2520
EXPECTED_OBSERVATIONS = 11_045_160

# ------------------------------------------------------------
# 2. Discover project root
# ------------------------------------------------------------

current_path = Path.cwd().resolve()
project_root = None

for candidate in [current_path, *current_path.parents]:
    expected_path = (
        candidate
        / "phase3b_weather_intelligence"
        / "data"
        / "processed"
        / "weather_daily_v1"
    )

    if expected_path.is_dir():
        project_root = candidate
        break

if project_root is None:
    raise FileNotFoundError(
        "Could not locate the BharatCrop project root.\n"
        "Expected:\n"
        "phase3b_weather_intelligence/data/processed/"
        "weather_daily_v1\n"
        f"Current directory: {current_path}"
    )

# ------------------------------------------------------------
# 3. Resolve input and output paths
# ------------------------------------------------------------

WEATHER_DATA_DIR = (
    project_root
    / "phase3b_weather_intelligence"
    / "data"
    / "processed"
    / "weather_daily_v1"
)

FEATURE_OUTPUT_DIR = (
    project_root
    / "phase3b_weather_intelligence"
    / "data"
    / "processed"
    / "weather_features_v1"
)

REPORT_DIR = (
    project_root
    / "phase3b_weather_intelligence"
    / "reports"
    / "weather_intelligence_v2"
)

FEATURE_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FEATURE_SPEC_PATH = (
    REPORT_DIR / "WI_03_feature_specification.json"
)

# ------------------------------------------------------------
# 4. Discover Parquet partitions
# ------------------------------------------------------------

parquet_paths = sorted(
    path
    for path in WEATHER_DATA_DIR.rglob("*.parquet")
    if path.is_file()
)

if not parquet_paths:
    raise FileNotFoundError(
        f"No Parquet partitions found in {WEATHER_DATA_DIR}"
    )

weather_dataset = ds.dataset(
    [str(path) for path in parquet_paths],
    format="parquet",
    partitioning="hive",
    partition_base_dir=str(WEATHER_DATA_DIR)
)

# ------------------------------------------------------------
# 5. Load required columns
# ------------------------------------------------------------

required_columns = [
    "cell_id",
    "latitude",
    "longitude",
    "date",
    "PRECTOTCORR",
    "T2M_MAX",
    "T2M_MIN",
    "qc_pass",
    "year"
]

weather = weather_dataset.to_table(
    columns=required_columns
).to_pandas()

weather["date"] = pd.to_datetime(
    weather["date"],
    errors="raise"
)

weather = weather.sort_values(
    ["cell_id", "date"]
).reset_index(drop=True)

# ------------------------------------------------------------
# 6. Environment and load summary
# ------------------------------------------------------------

print("=" * 65)
print("WI-03 ENVIRONMENT INITIALIZED")
print("=" * 65)

print("Python:", platform.python_version())
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("PyArrow:", pa.__version__)

print("\nProject root:", project_root)
print("Input dataset:", WEATHER_DATA_DIR)
print("Parquet partitions:", len(parquet_paths))

print("\nLoaded observations:", f"{len(weather):,}")
print("Unique grid cells:", weather["cell_id"].nunique())
print("Date range:", weather["date"].min(), "to", weather["date"].max())

print("\nDevelopment period:")
print(DEVELOPMENT_START.date(), "to", DEVELOPMENT_END.date())

print("\nFinal holdout period:")
print(FINAL_HOLDOUT_START.date(), "to", FINAL_HOLDOUT_END.date())

print("\nForecast horizons:", FORECAST_HORIZONS)
print("Lag days:", LAG_DAYS)
print("Rolling windows:", ROLLING_WINDOWS)

print("\nFeature output directory:", FEATURE_OUTPUT_DIR)
print("Report directory:", REPORT_DIR)

print("\nDataset loaded successfully.")


WI-03 ENVIRONMENT INITIALIZED
Python: 3.13.5
Pandas: 2.3.3
NumPy: 2.5.4
PyArrow: 26.0.1

Project root: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence
Input dataset: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\data\processed\weather_daily_v1
Parquet partitions: 12

Loaded observations: 11,045,160
Unique grid cells: 2520
Date range: 2013-01-01 00:00:00 to 2024-12-31 00:00:00

Development period:
2013-01-01 to 2022-12-31

Final holdout period:
2023-01-01 to 2024-12-31

Forecast horizons: [1, 2, 3, 4, 5, 6, 7]
Lag days: [1, 2, 3, 7, 14, 30]
Rolling windows: [3, 7, 14, 30]

Feature output directory: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\data\processed\weather_features_v1
Report directory: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2

Dataset loaded successfully.


In [2]:

# ============================================================
# WI-03 | CELL 3: STRUCTURAL VALIDATION AND FEATURE SPECIFICATION
# ============================================================

# ------------------------------------------------------------
# 1. Validate required columns and data types
# ------------------------------------------------------------

required_columns = [
    "cell_id",
    "latitude",
    "longitude",
    "date",
    "PRECTOTCORR",
    "T2M_MAX",
    "T2M_MIN",
    "qc_pass",
    "year"
]

missing_columns = sorted(
    set(required_columns) - set(weather.columns)
)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

if not pd.api.types.is_datetime64_any_dtype(weather["date"]):
    raise TypeError("date must be a pandas datetime column.")

numeric_columns = [
    "latitude",
    "longitude",
    "PRECTOTCORR",
    "T2M_MAX",
    "T2M_MIN"
]

for column in numeric_columns:
    if not pd.api.types.is_numeric_dtype(weather[column]):
        raise TypeError(
            f"{column} must be numeric."
        )

# ------------------------------------------------------------
# 2. Validate missing and non-finite values
# ------------------------------------------------------------

missing_summary = (
    weather[required_columns]
    .isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

non_finite_summary = {}

for column in numeric_columns:
    non_finite_summary[column] = int(
        (~np.isfinite(weather[column])).sum()
    )

print("=" * 65)
print("MISSING VALUE SUMMARY")
print("=" * 65)
display(missing_summary)

print("\nNON-FINITE NUMERIC VALUES")
display(
    pd.Series(
        non_finite_summary,
        name="non_finite_count"
    ).to_frame()
)

if missing_summary["missing_count"].sum() > 0:
    raise ValueError(
        "Missing values detected. Investigate before feature engineering."
    )

if any(value > 0 for value in non_finite_summary.values()):
    raise ValueError(
        "Non-finite numeric values detected."
    )

# ------------------------------------------------------------
# 3. Validate QC and physical constraints
# ------------------------------------------------------------

qc_summary = (
    weather["qc_pass"]
    .value_counts(dropna=False)
    .rename_axis("qc_pass")
    .to_frame("row_count")
)

print("\nQC STATUS")
display(qc_summary)

if not weather["qc_pass"].all():
    raise ValueError(
        "Rows failing stored QC were found."
    )

negative_rainfall_count = int(
    (weather["PRECTOTCORR"] < 0).sum()
)

invalid_temperature_order_count = int(
    (weather["T2M_MIN"] > weather["T2M_MAX"]).sum()
)

if negative_rainfall_count:
    raise ValueError(
        f"Negative rainfall observations: {negative_rainfall_count}"
    )

if invalid_temperature_order_count:
    raise ValueError(
        "T2M_MIN exceeds T2M_MAX for some observations."
    )

# ------------------------------------------------------------
# 4. Validate canonical keys
# ------------------------------------------------------------

duplicate_key_count = int(
    weather.duplicated(
        ["cell_id", "date"]
    ).sum()
)

print("\nDuplicate (cell_id, date) keys:", duplicate_key_count)

if duplicate_key_count:
    raise ValueError(
        "Duplicate grid-cell/date keys detected."
    )

# ------------------------------------------------------------
# 5. Validate grid-cell coordinates
# ------------------------------------------------------------

coordinate_counts = (
    weather.groupby("cell_id")
    .agg(
        latitude_count=("latitude", "nunique"),
        longitude_count=("longitude", "nunique")
    )
)

invalid_coordinate_cells = coordinate_counts[
    (coordinate_counts["latitude_count"] != 1)
    | (coordinate_counts["longitude_count"] != 1)
]

if not invalid_coordinate_cells.empty:
    raise ValueError(
        "Some grid cells have inconsistent coordinates."
    )

# ------------------------------------------------------------
# 6. Validate daily continuity for every grid cell
# ------------------------------------------------------------

expected_dates = pd.date_range(
    START_DATE,
    END_DATE,
    freq="D"
)

cell_date_summary = (
    weather.groupby("cell_id")
    .agg(
        first_date=("date", "min"),
        last_date=("date", "max"),
        unique_dates=("date", "nunique"),
        row_count=("date", "size")
    )
)

cell_date_summary["expected_dates"] = len(expected_dates)

cell_date_summary["continuous"] = (
    (cell_date_summary["first_date"] == START_DATE)
    & (cell_date_summary["last_date"] == END_DATE)
    & (cell_date_summary["unique_dates"] == len(expected_dates))
    & (cell_date_summary["row_count"] == len(expected_dates))
)

print("\nGRID-CELL DAILY CONTINUITY")
display(
    cell_date_summary[
        [
            "first_date",
            "last_date",
            "unique_dates",
            "row_count",
            "continuous"
        ]
    ].describe(include="all")
)

non_continuous_cells = int(
    (~cell_date_summary["continuous"]).sum()
)

print("\nNon-continuous grid cells:", non_continuous_cells)

if non_continuous_cells:
    raise ValueError(
        "Daily continuity failed for one or more grid cells."
    )

# ------------------------------------------------------------
# 7. Validate chronological development and holdout boundaries
# ------------------------------------------------------------

development_rows = int(
    weather["date"].between(
        DEVELOPMENT_START,
        DEVELOPMENT_END
    ).sum()
)

holdout_rows = int(
    weather["date"].between(
        FINAL_HOLDOUT_START,
        FINAL_HOLDOUT_END
    ).sum()
)

outside_split_rows = len(weather) - (
    development_rows + holdout_rows
)

split_summary = pd.DataFrame({
    "period": [
        "Development",
        "Final holdout",
        "Outside configured periods"
    ],
    "start_date": [
        DEVELOPMENT_START,
        FINAL_HOLDOUT_START,
        None
    ],
    "end_date": [
        DEVELOPMENT_END,
        FINAL_HOLDOUT_END,
        None
    ],
    "row_count": [
        development_rows,
        holdout_rows,
        outside_split_rows
    ]
})

print("\nCHRONOLOGICAL PERIOD SUMMARY")
display(split_summary)

if outside_split_rows != 0:
    raise ValueError(
        "Some observations fall outside the configured periods."
    )

# ------------------------------------------------------------
# 8. Define feature-generation policy
# ------------------------------------------------------------

feature_specification = {
    "notebook": "WI-03",
    "dataset_version": "weather-daily-v1",
    "forecast_targets": {
        "PRECTOTCORR": "Daily precipitation (mm/day)",
        "T2M_MAX": "Daily maximum temperature (C)"
    },
    "input_variables": WEATHER_VARIABLES,
    "forecast_horizons_days": FORECAST_HORIZONS,
    "lag_days": LAG_DAYS,
    "rolling_windows_days": ROLLING_WINDOWS,
    "rolling_policy": (
        "Shift each input series by one day before calculating "
        "rolling statistics. Features at date t must not include "
        "weather observed after t."
    ),
    "calendar_features": [
        "day_of_year",
        "month",
        "day_of_week",
        "day_of_year_sin",
        "day_of_year_cos",
        "day_of_week_sin",
        "day_of_week_cos"
    ],
    "spatial_features": [
        "cell_id",
        "latitude",
        "longitude"
    ],
    "development_period": {
        "start": str(DEVELOPMENT_START.date()),
        "end": str(DEVELOPMENT_END.date())
    },
    "final_holdout_period": {
        "start": str(FINAL_HOLDOUT_START.date()),
        "end": str(FINAL_HOLDOUT_END.date())
    },
    "split_policy": (
        "Assign samples by target date, not feature-origin date. "
        "Historical inputs before a target date may cross the "
        "development/holdout boundary."
    ),
    "prohibited_inputs": [
        "APY variables",
        "Crop and yield variables",
        "Future weather observations",
        "Full-period climatology",
        "Full-period anomaly values"
    ],
    "model_training_performed": False
}

with open(
    FEATURE_SPEC_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        feature_specification,
        file,
        indent=2,
        ensure_ascii=False
    )

print("\nFeature specification saved to:")
print(FEATURE_SPEC_PATH)

print("\nAll WI-03 structural validation checks passed.")


MISSING VALUE SUMMARY


,missing_count
cell_id,0
latitude,0
longitude,0
date,0
PRECTOTCORR,0
T2M_MAX,0
T2M_MIN,0
qc_pass,0
year,0



NON-FINITE NUMERIC VALUES


,non_finite_count
latitude,0
longitude,0
PRECTOTCORR,0
T2M_MAX,0
T2M_MIN,0



QC STATUS


,row_count
qc_pass,
True,11045160



Duplicate (cell_id, date) keys: 0

GRID-CELL DAILY CONTINUITY


,first_date,last_date,unique_dates,row_count,continuous
count,2520,2520,2520.0,2520.0,2520
unique,NaN,NaN,NaN,NaN,1
top,NaN,NaN,NaN,NaN,True
freq,NaN,NaN,NaN,NaN,2520
mean,2013-01-01 00:00:00,2024-12-31 00:00:00,4383.0,4383.0,NaN
min,2013-01-01 00:00:00,2024-12-31 00:00:00,4383.0,4383.0,NaN
25%,2013-01-01 00:00:00,2024-12-31 00:00:00,4383.0,4383.0,NaN
50%,2013-01-01 00:00:00,2024-12-31 00:00:00,4383.0,4383.0,NaN
75%,2013-01-01 00:00:00,2024-12-31 00:00:00,4383.0,4383.0,NaN
max,2013-01-01 00:00:00,2024-12-31 00:00:00,4383.0,4383.0,NaN



Non-continuous grid cells: 0

CHRONOLOGICAL PERIOD SUMMARY


,period,start_date,end_date,row_count
0,Development,2013-01-01,2022-12-31,9203040
1,Final holdout,2023-01-01,2024-12-31,1842120
2,Outside configured periods,NaT,NaT,0



Feature specification saved to:
C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_03_feature_specification.json

All WI-03 structural validation checks passed.


In [3]:

# ============================================================
# WI-03 | CELL 4: FEATURE ENGINEERING PROTOTYPE
# ============================================================

# Select a small, deterministic sample of grid cells
prototype_cells = sorted(
    weather["cell_id"].unique()
)[:3]

prototype = (
    weather[
        weather["cell_id"].isin(prototype_cells)
    ]
    .sort_values(["cell_id", "date"])
    .copy()
    .reset_index(drop=True)
)

print("=" * 65)
print("FEATURE ENGINEERING PROTOTYPE")
print("=" * 65)
print("Selected cells:", prototype_cells)
print("Prototype observations:", f"{len(prototype):,}")

# ------------------------------------------------------------
# 1. Create calendar features
# ------------------------------------------------------------

prototype["day_of_year"] = prototype["date"].dt.dayofyear
prototype["month"] = prototype["date"].dt.month
prototype["day_of_week"] = prototype["date"].dt.dayofweek

prototype["day_of_year_sin"] = np.sin(
    2 * np.pi * prototype["day_of_year"] / 365.25
)
prototype["day_of_year_cos"] = np.cos(
    2 * np.pi * prototype["day_of_year"] / 365.25
)

prototype["day_of_week_sin"] = np.sin(
    2 * np.pi * prototype["day_of_week"] / 7
)
prototype["day_of_week_cos"] = np.cos(
    2 * np.pi * prototype["day_of_week"] / 7
)

# ------------------------------------------------------------
# 2. Generate lag features
# ------------------------------------------------------------

for variable in WEATHER_VARIABLES:
    grouped = prototype.groupby("cell_id")[variable]

    for lag in LAG_DAYS:
        prototype[f"{variable}_lag_{lag}"] = (
            grouped.shift(lag)
        )

# ------------------------------------------------------------
# 3. Generate rolling features
# ------------------------------------------------------------

for variable in WEATHER_VARIABLES:
    grouped = prototype.groupby("cell_id")[variable]

    for window in ROLLING_WINDOWS:
        prototype[f"{variable}_rolling_mean_{window}"] = (
            grouped.transform(
                lambda series: series.rolling(
                    window=window,
                    min_periods=window
                ).mean()
            )
        )

        prototype[f"{variable}_rolling_std_{window}"] = (
            grouped.transform(
                lambda series: series.rolling(
                    window=window,
                    min_periods=window
                ).std()
            )
        )

# ------------------------------------------------------------
# 4. Generate multi-horizon targets
# ------------------------------------------------------------

for target in TARGET_VARIABLES:
    grouped = prototype.groupby("cell_id")[target]

    for horizon in FORECAST_HORIZONS:
        prototype[f"{target}_target_t_plus_{horizon}"] = (
            grouped.shift(-horizon)
        )

# ------------------------------------------------------------
# 5. Inspect generated features
# ------------------------------------------------------------

generated_feature_columns = [
    column
    for column in prototype.columns
    if "_lag_" in column
    or "_rolling_" in column
]

generated_target_columns = [
    column
    for column in prototype.columns
    if "_target_t_plus_" in column
]

print("\nGenerated lag and rolling features:",
      len(generated_feature_columns))
print("Generated target columns:",
      len(generated_target_columns))

print("\nSample feature rows")
display(
    prototype[
        [
            "cell_id",
            "date",
            "PRECTOTCORR",
            "PRECTOTCORR_lag_1",
            "PRECTOTCORR_lag_7",
            "PRECTOTCORR_rolling_mean_3",
            "PRECTOTCORR_target_t_plus_1",
            "T2M_MAX",
            "T2M_MAX_lag_1",
            "T2M_MAX_rolling_mean_7",
            "T2M_MAX_target_t_plus_1"
        ]
    ].iloc[28:35]
)

# ------------------------------------------------------------
# 6. Validate feature alignment
# ------------------------------------------------------------

sample_cell = prototype_cells[0]

sample_rows = prototype[
    prototype["cell_id"] == sample_cell
].reset_index(drop=True)

test_index = 100

expected_lag_1 = sample_rows.loc[
    test_index, "PRECTOTCORR"
]

actual_lag_1 = sample_rows.loc[
    test_index + 1, "PRECTOTCORR_lag_1"
]

expected_target_t_plus_1 = sample_rows.loc[
    test_index + 1, "PRECTOTCORR"
]

actual_target_t_plus_1 = sample_rows.loc[
    test_index, "PRECTOTCORR_target_t_plus_1"
]

assert np.isclose(expected_lag_1, actual_lag_1)
assert np.isclose(
    expected_target_t_plus_1,
    actual_target_t_plus_1
)

print("\nLag alignment check: PASSED")
print("T+1 target alignment check: PASSED")

# ------------------------------------------------------------
# 7. Validate no cross-cell contamination
# ------------------------------------------------------------

for cell_id in prototype_cells:
    cell_data = prototype[
        prototype["cell_id"] == cell_id
    ]

    assert len(cell_data) == 4383
    assert cell_data["date"].is_monotonic_increasing

print("Grid-cell grouping check: PASSED")

print("\nPrototype feature engineering completed.")


FEATURE ENGINEERING PROTOTYPE
Selected cells: ['cell_0000', 'cell_0001', 'cell_0002']
Prototype observations: 13,149

Generated lag and rolling features: 42
Generated target columns: 14

Sample feature rows


,cell_id,date,PRECTOTCORR,PRECTOTCORR_lag_1,PRECTOTCORR_lag_7,PRECTOTCORR_rolling_mean_3,PRECTOTCORR_target_t_plus_1,T2M_MAX,T2M_MAX_lag_1,T2M_MAX_rolling_mean_7,T2M_MAX_target_t_plus_1
28,cell_0000,2013-01-29,0.05,0.08,0.03,0.063333,0.02,27.78,27.22,27.944286,28.09
29,cell_0000,2013-01-30,0.02,0.05,0.07,0.050000,0.10,28.09,27.78,27.898571,27.83
30,cell_0000,2013-01-31,0.10,0.02,0.03,0.056667,0.09,27.83,28.09,27.855714,27.72
31,cell_0000,2013-02-01,0.09,0.10,0.01,0.070000,0.07,27.72,27.83,27.832857,27.40
32,cell_0000,2013-02-02,0.07,0.09,0.00,0.086667,0.06,27.40,27.72,27.721429,27.44
33,cell_0000,2013-02-03,0.06,0.07,0.06,0.073333,0.10,27.44,27.40,27.640000,27.00
34,cell_0000,2013-02-04,0.10,0.06,0.08,0.076667,0.15,27.00,27.44,27.608571,27.33



Lag alignment check: PASSED
T+1 target alignment check: PASSED
Grid-cell grouping check: PASSED

Prototype feature engineering completed.


In [4]:

# ============================================================
# WI-03 — CELL 4A
# Explicitly identify weather observations available at origin t
# ============================================================

CURRENT_FEATURE_COLUMNS = []

for variable in WEATHER_VARIABLES:
    current_column = f"{variable}_current"

    prototype[current_column] = prototype[variable]
    CURRENT_FEATURE_COLUMNS.append(current_column)

# Verify current-day features match the source observations.
for variable in WEATHER_VARIABLES:
    current_column = f"{variable}_current"

    assert np.allclose(
        prototype[current_column].to_numpy(),
        prototype[variable].to_numpy(),
        equal_nan=True
    ), f"Current feature mismatch: {variable}"

# Verify rolling windows include the forecast-origin day.
sample_cell = prototype["cell_id"].iloc[0]
sample_date = prototype.loc[100, "date"]

sample_rows = prototype[
    prototype["cell_id"] == sample_cell
].reset_index(drop=True)

sample_position = 100

expected_rolling_3 = sample_rows.loc[
    sample_position - 2:sample_position,
    "PRECTOTCORR"
].mean()

actual_rolling_3 = sample_rows.loc[
    sample_position,
    "PRECTOTCORR_rolling_mean_3"
]

assert np.isclose(expected_rolling_3, actual_rolling_3)

print("Current-day feature columns:", CURRENT_FEATURE_COLUMNS)
print("Current observation checks: PASSED")
print("Rolling-window origin alignment: PASSED")
print("Forecast origin convention: features available through day t")

Current-day feature columns: ['PRECTOTCORR_current', 'T2M_MAX_current', 'T2M_MIN_current']
Current observation checks: PASSED
Rolling-window origin alignment: PASSED
Forecast origin convention: features available through day t


In [5]:

# ============================================================
# WI-03 — CELL 4B
# Update and persist the forecast-origin convention
# ============================================================

import json

with open(FEATURE_SPEC_PATH, "r", encoding="utf-8") as file:
    feature_spec = json.load(file)

feature_spec["forecast_origin_convention"] = {
    "forecast_origin": "End of day t, after day t weather observations are available",
    "current_features": (
        "Weather observations on day t, explicitly named with _current"
    ),
    "lag_features": (
        "lag_1 means day t-1; lag_k means day t-k"
    ),
    "rolling_features": (
        "Rolling windows end on day t and include the observation on day t"
    ),
    "targets": (
        "T_plus_h means the target observation on day t+h"
    ),
    "availability_rule": (
        "Every predictor must be available at the forecast origin"
    )
}

with open(FEATURE_SPEC_PATH, "w", encoding="utf-8") as file:
    json.dump(feature_spec, file, indent=2, ensure_ascii=False)

print("Updated feature specification:", FEATURE_SPEC_PATH)
print(json.dumps(
    feature_spec["forecast_origin_convention"],
    indent=2,
    ensure_ascii=False
))


Updated feature specification: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_03_feature_specification.json
{
  "forecast_origin": "End of day t, after day t weather observations are available",
  "current_features": "Weather observations on day t, explicitly named with _current",
  "lag_features": "lag_1 means day t-1; lag_k means day t-k",
  "rolling_features": "Rolling windows end on day t and include the observation on day t",
  "targets": "T_plus_h means the target observation on day t+h",
  "availability_rule": "Every predictor must be available at the forecast origin"
}


In [6]:

# ============================================================
# WI-03 — CELL 5
# Estimate full-dataset feature dimensions and memory
# ============================================================

import pandas as pd

base_rows = len(weather)

current_feature_count = len(WEATHER_VARIABLES)
lag_feature_count = len(WEATHER_VARIABLES) * len(LAG_DAYS)
rolling_feature_count = (
    len(WEATHER_VARIABLES) * len(ROLLING_WINDOWS) * 2
)
calendar_feature_count = 6
spatial_feature_count = 3
target_column_count = len(TARGET_VARIABLES) * len(FORECAST_HORIZONS)

total_predictor_count = (
    current_feature_count
    + lag_feature_count
    + rolling_feature_count
    + calendar_feature_count
    + spatial_feature_count
)

estimated_columns = (
    2  # cell_id and date
    + total_predictor_count
    + target_column_count
)

# Estimate the in-memory size of the source DataFrame.
source_memory_mb = weather.memory_usage(
    index=True, deep=True
).sum() / (1024 ** 2)

# Approximate engineered numeric storage using float64.
estimated_numeric_mb = (
    base_rows
    * (total_predictor_count + target_column_count)
    * 8
    / (1024 ** 2)
)

print("Base observations:", f"{base_rows:,}")
print("Current features:", current_feature_count)
print("Lag features:", lag_feature_count)
print("Rolling mean/std features:", rolling_feature_count)
print("Calendar features:", calendar_feature_count)
print("Spatial features:", spatial_feature_count)
print("Total predictors:", total_predictor_count)
print("Target columns:", target_column_count)
print("Estimated total columns:", estimated_columns)
print(f"Source DataFrame memory: {source_memory_mb:,.1f} MiB")
print(f"Approx. engineered numeric storage: {estimated_numeric_mb:,.1f} MiB")
print(
    "Approx. combined memory before temporary copies:",
    f"{source_memory_mb + estimated_numeric_mb:,.1f} MiB"
)


Base observations: 11,045,160
Current features: 3
Lag features: 18
Rolling mean/std features: 24
Calendar features: 6
Spatial features: 3
Total predictors: 54
Target columns: 14
Estimated total columns: 70
Source DataFrame memory: 1,169.2 MiB
Approx. engineered numeric storage: 5,730.2 MiB
Approx. combined memory before temporary copies: 6,899.4 MiB


In [8]:

from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence"
).resolve()

assert PROJECT_ROOT.exists(), (
    f"Project root does not exist: {PROJECT_ROOT}"
)

assert (PROJECT_ROOT / "phase3b_weather_intelligence").exists(), (
    "phase3b_weather_intelligence directory not found."
)

FEATURE_OUTPUT_DIR = (
    PROJECT_ROOT
    / "phase3b_weather_intelligence"
    / "data"
    / "processed"
    / "weather_features_v1"
)

print("Project root:", PROJECT_ROOT)
print("Feature output directory:", FEATURE_OUTPUT_DIR)
print("Project root verified:", PROJECT_ROOT.is_dir())


Project root: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence
Feature output directory: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\data\processed\weather_features_v1
Project root verified: True


In [9]:

# ============================================================
# WI-03 — CELL 6
# Inspect feature output directory before creating artifacts
# ============================================================

from pathlib import Path
import shutil

FEATURE_OUTPUT_DIR = (
    PROJECT_ROOT
    / "phase3b_weather_intelligence"
    / "data"
    / "processed"
    / "weather_features_v1"
)

FEATURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

existing_files = sorted(
    path for path in FEATURE_OUTPUT_DIR.rglob("*")
    if path.is_file()
)

disk = shutil.disk_usage(PROJECT_ROOT)

print("Feature output directory:", FEATURE_OUTPUT_DIR)
print("Existing files:", len(existing_files))

for path in existing_files[:30]:
    print(" -", path.relative_to(FEATURE_OUTPUT_DIR))

if len(existing_files) > 30:
    print(f"... and {len(existing_files) - 30} more")

print(f"Free disk space: {disk.free / (1024 ** 3):.2f} GiB")
print(f"Total disk space: {disk.total / (1024 ** 3):.2f} GiB")

if existing_files:
    print(
        "\nSTOP: Output directory is not empty. "
        "Do not write or overwrite files until existing artifacts are reviewed."
    )
else:
    print("\nOutput directory is empty. Safe to plan a fresh feature build.")


Feature output directory: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\data\processed\weather_features_v1
Existing files: 0
Free disk space: 10.05 GiB
Total disk space: 475.78 GiB

Output directory is empty. Safe to plan a fresh feature build.


In [13]:

# ============================================================
# WI-03 — CELL 7A
# Inspect actual weather dataset schema
# ============================================================

print("Weather columns:")
for column in weather.columns:
    print(f" - {column}: {weather[column].dtype}")

print("\nPotential coordinate columns:")
coordinate_candidates = [
    column for column in weather.columns
    if any(term in column.lower() for term in ["lat", "lon", "longitude", "latitude"])
]
print(coordinate_candidates)

print("\nSample rows:")
display(weather.head(3))


Weather columns:
 - cell_id: object
 - latitude: float64
 - longitude: float64
 - date: datetime64[ns]
 - PRECTOTCORR: float64
 - T2M_MAX: float64
 - T2M_MIN: float64
 - qc_pass: bool
 - year: int32

Potential coordinate columns:
['latitude', 'longitude']

Sample rows:


,cell_id,latitude,longitude,date,PRECTOTCORR,T2M_MAX,T2M_MIN,qc_pass,year
0,cell_0000,7.0,69.375,2013-01-01,1.63,28.39,27.01,True,2013
1,cell_0000,7.0,69.375,2013-01-02,0.63,27.45,27.06,True,2013
2,cell_0000,7.0,69.375,2013-01-03,0.27,27.81,27.08,True,2013


In [14]:

# ============================================================
# WI-03 — CELL 7B
# Pilot feature engineering: 5 grid cells
# ============================================================

import numpy as np
import pandas as pd

BATCH_CELL_COUNT = 5
all_cells = sorted(weather["cell_id"].unique())
pilot_cells = all_cells[:BATCH_CELL_COUNT]

pilot = (
    weather[weather["cell_id"].isin(pilot_cells)]
    .sort_values(["cell_id", "date"])
    .copy()
    .reset_index(drop=True)
)

# Current-day predictors.
for variable in WEATHER_VARIABLES:
    pilot[f"{variable}_current"] = pilot[variable]

# Calendar predictors.
pilot["day_of_year"] = pilot["date"].dt.dayofyear
pilot["month"] = pilot["date"].dt.month
pilot["day_of_week"] = pilot["date"].dt.dayofweek

pilot["day_of_year_sin"] = np.sin(
    2 * np.pi * pilot["day_of_year"] / 365.25
)
pilot["day_of_year_cos"] = np.cos(
    2 * np.pi * pilot["day_of_year"] / 365.25
)
pilot["day_of_week_sin"] = np.sin(
    2 * np.pi * pilot["day_of_week"] / 7
)

# Spatial predictors: actual canonical column names.
coordinate_counts = (
    pilot.groupby("cell_id")[["latitude", "longitude"]]
    .nunique()
)

assert (coordinate_counts <= 1).all().all(), (
    "A grid cell has inconsistent coordinates."
)

# Lag and rolling predictors.
for variable in WEATHER_VARIABLES:
    grouped = pilot.groupby("cell_id", sort=False)[variable]

    for lag in LAG_DAYS:
        pilot[f"{variable}_lag_{lag}"] = grouped.shift(lag)

    for window in ROLLING_WINDOWS:
        pilot[f"{variable}_rolling_mean_{window}"] = (
            grouped.transform(
                lambda series: series.rolling(
                    window=window,
                    min_periods=window
                ).mean()
            )
        )

        pilot[f"{variable}_rolling_std_{window}"] = (
            grouped.transform(
                lambda series: series.rolling(
                    window=window,
                    min_periods=window
                ).std()
            )
        )

# Forecast targets.
for target in TARGET_VARIABLES:
    grouped = pilot.groupby("cell_id", sort=False)[target]

    for horizon in FORECAST_HORIZONS:
        pilot[f"{target}_target_t_plus_{horizon}"] = (
            grouped.shift(-horizon)
        )

# Validation.
expected_rows = BATCH_CELL_COUNT * len(weather["date"].unique())
assert len(pilot) == expected_rows

cell_rows = (
    pilot[pilot["cell_id"] == pilot_cells[0]]
    .reset_index(drop=True)
)

test_position = 100

assert np.isclose(
    cell_rows.loc[test_position, "PRECTOTCORR_target_t_plus_1"],
    cell_rows.loc[test_position + 1, "PRECTOTCORR"]
)

assert np.isclose(
    cell_rows.loc[test_position, "T2M_MAX_target_t_plus_7"],
    cell_rows.loc[test_position + 7, "T2M_MAX"]
)

assert (
    pilot.groupby("cell_id")[["latitude", "longitude"]]
    .nunique()
    .le(1)
    .all()
    .all()
)

print("Pilot cells:", pilot_cells)
print("Pilot rows:", f"{len(pilot):,}")
print("Pilot columns:", len(pilot.columns))
print(
    "Pilot memory:",
    f"{pilot.memory_usage(deep=True).sum() / 1024**2:.1f} MiB"
)
print("T+1 and T+7 alignment: PASSED")
print("Coordinate consistency: PASSED")
print("\nMissing values by column (top 10):")
print(pilot.isna().sum().sort_values(ascending=False).head(10))


Pilot cells: ['cell_0000', 'cell_0001', 'cell_0002', 'cell_0003', 'cell_0004']
Pilot rows: 21,915
Pilot columns: 74
Pilot memory: 12.9 MiB
T+1 and T+7 alignment: PASSED
Coordinate consistency: PASSED

Missing values by column (top 10):
T2M_MAX_lag_30                 150
PRECTOTCORR_lag_30             150
T2M_MIN_lag_30                 150
PRECTOTCORR_rolling_mean_30    145
T2M_MAX_rolling_mean_30        145
T2M_MIN_rolling_mean_30        145
PRECTOTCORR_rolling_std_30     145
T2M_MAX_rolling_std_30         145
T2M_MIN_rolling_std_30         145
T2M_MAX_lag_14                  70
dtype: int64


In [15]:

# ============================================================
# WI-03 — CELL 8
# Define the canonical feature and target schema
# ============================================================

ID_COLUMNS = ["cell_id", "date"]

SPATIAL_FEATURE_COLUMNS = ["latitude", "longitude"]

CURRENT_FEATURE_COLUMNS = [
    f"{variable}_current"
    for variable in WEATHER_VARIABLES
]

LAG_FEATURE_COLUMNS = [
    f"{variable}_lag_{lag}"
    for variable in WEATHER_VARIABLES
    for lag in LAG_DAYS
]

ROLLING_FEATURE_COLUMNS = [
    f"{variable}_rolling_{stat}_{window}"
    for variable in WEATHER_VARIABLES
    for window in ROLLING_WINDOWS
    for stat in ["mean", "std"]
]

CALENDAR_FEATURE_COLUMNS = [
    "day_of_year",
    "month",
    "day_of_week",
    "day_of_year_sin",
    "day_of_year_cos",
    "day_of_week_sin",
]

FEATURE_COLUMNS = (
    SPATIAL_FEATURE_COLUMNS
    + CURRENT_FEATURE_COLUMNS
    + LAG_FEATURE_COLUMNS
    + ROLLING_FEATURE_COLUMNS
    + CALENDAR_FEATURE_COLUMNS
)

TARGET_COLUMNS = [
    f"{target}_target_t_plus_{horizon}"
    for target in TARGET_VARIABLES
    for horizon in FORECAST_HORIZONS
]

EXPECTED_COLUMNS = ID_COLUMNS + FEATURE_COLUMNS + TARGET_COLUMNS

print("ID columns:", len(ID_COLUMNS))
print("Predictor columns:", len(FEATURE_COLUMNS))
print("Target columns:", len(TARGET_COLUMNS))
print("Total canonical columns:", len(EXPECTED_COLUMNS))

print("\nFeature columns:")
print(FEATURE_COLUMNS)

print("\nTarget columns:")
print(TARGET_COLUMNS)

assert len(FEATURE_COLUMNS) == len(set(FEATURE_COLUMNS))
assert len(TARGET_COLUMNS) == len(set(TARGET_COLUMNS))
assert not set(FEATURE_COLUMNS).intersection(TARGET_COLUMNS)
assert set(EXPECTED_COLUMNS).issubset(pilot.columns)

print("\nCanonical schema validation: PASSED")


ID columns: 2
Predictor columns: 53
Target columns: 14
Total canonical columns: 69

Feature columns:
['latitude', 'longitude', 'PRECTOTCORR_current', 'T2M_MAX_current', 'T2M_MIN_current', 'PRECTOTCORR_lag_1', 'PRECTOTCORR_lag_2', 'PRECTOTCORR_lag_3', 'PRECTOTCORR_lag_7', 'PRECTOTCORR_lag_14', 'PRECTOTCORR_lag_30', 'T2M_MAX_lag_1', 'T2M_MAX_lag_2', 'T2M_MAX_lag_3', 'T2M_MAX_lag_7', 'T2M_MAX_lag_14', 'T2M_MAX_lag_30', 'T2M_MIN_lag_1', 'T2M_MIN_lag_2', 'T2M_MIN_lag_3', 'T2M_MIN_lag_7', 'T2M_MIN_lag_14', 'T2M_MIN_lag_30', 'PRECTOTCORR_rolling_mean_3', 'PRECTOTCORR_rolling_std_3', 'PRECTOTCORR_rolling_mean_7', 'PRECTOTCORR_rolling_std_7', 'PRECTOTCORR_rolling_mean_14', 'PRECTOTCORR_rolling_std_14', 'PRECTOTCORR_rolling_mean_30', 'PRECTOTCORR_rolling_std_30', 'T2M_MAX_rolling_mean_3', 'T2M_MAX_rolling_std_3', 'T2M_MAX_rolling_mean_7', 'T2M_MAX_rolling_std_7', 'T2M_MAX_rolling_mean_14', 'T2M_MAX_rolling_std_14', 'T2M_MAX_rolling_mean_30', 'T2M_MAX_rolling_std_30', 'T2M_MIN_rolling_mean_3', 'T

In [16]:

# ============================================================
# WI-03 — CELL 9
# Validate exact canonical pilot dataset
# ============================================================

pilot_final = pilot[EXPECTED_COLUMNS].copy()

# Confirm exact schema and order.
assert pilot_final.columns.tolist() == EXPECTED_COLUMNS
assert len(pilot_final.columns) == 69

# Confirm unique observation keys.
assert not pilot_final.duplicated(["cell_id", "date"]).any()

# Confirm no raw weather columns or QC helper columns leaked in.
assert not any(
    column in pilot_final.columns
    for column in WEATHER_VARIABLES + ["qc_pass", "year"]
)

# Separate feature and target missingness.
feature_missing = pilot_final[FEATURE_COLUMNS].isna().sum()
target_missing = pilot_final[TARGET_COLUMNS].isna().sum()

print("Canonical pilot rows:", f"{len(pilot_final):,}")
print("Canonical pilot columns:", len(pilot_final.columns))
print(
    "Canonical pilot memory:",
    f"{pilot_final.memory_usage(deep=True).sum() / 1024**2:.2f} MiB"
)

print("\nFeature missingness:")
print(feature_missing[feature_missing > 0].to_string() or "None")

print("\nTarget missingness:")
print(target_missing[target_missing > 0].to_string() or "None")

print("\nMissing feature rows:", int(pilot_final[FEATURE_COLUMNS].isna().any(axis=1).sum()))
print("Missing target rows:", int(pilot_final[TARGET_COLUMNS].isna().any(axis=1).sum()))

print("\nExact schema validation: PASSED")


Canonical pilot rows: 21,915
Canonical pilot columns: 69
Canonical pilot memory: 12.33 MiB

Feature missingness:
PRECTOTCORR_lag_1                5
PRECTOTCORR_lag_2               10
PRECTOTCORR_lag_3               15
PRECTOTCORR_lag_7               35
PRECTOTCORR_lag_14              70
PRECTOTCORR_lag_30             150
T2M_MAX_lag_1                    5
T2M_MAX_lag_2                   10
T2M_MAX_lag_3                   15
T2M_MAX_lag_7                   35
T2M_MAX_lag_14                  70
T2M_MAX_lag_30                 150
T2M_MIN_lag_1                    5
T2M_MIN_lag_2                   10
T2M_MIN_lag_3                   15
T2M_MIN_lag_7                   35
T2M_MIN_lag_14                  70
T2M_MIN_lag_30                 150
PRECTOTCORR_rolling_mean_3      10
PRECTOTCORR_rolling_std_3       10
PRECTOTCORR_rolling_mean_7      30
PRECTOTCORR_rolling_std_7       30
PRECTOTCORR_rolling_mean_14     65
PRECTOTCORR_rolling_std_14      65
PRECTOTCORR_rolling_mean_30    145
PRECTOTCORR_

In [17]:

# ============================================================
# WI-03 — CELL 10
# Configure incremental feature dataset construction
# ============================================================

import pyarrow as pa
import pyarrow.parquet as pq

BATCH_CELL_COUNT = 10
PARQUET_COMPRESSION = "zstd"
PARQUET_COMPRESSION_LEVEL = 3

FEATURE_DATASET_VERSION = "weather-features-v1"
FEATURE_DATASET_DIR = FEATURE_OUTPUT_DIR / FEATURE_DATASET_VERSION

# Do not overwrite an existing version.
if FEATURE_DATASET_DIR.exists() and any(FEATURE_DATASET_DIR.iterdir()):
    raise FileExistsError(
        f"Output version already contains files: {FEATURE_DATASET_DIR}. "
        "Choose a new version rather than overwriting existing artifacts."
    )

FEATURE_DATASET_DIR.mkdir(parents=True, exist_ok=True)

# Calculate the expected batch size.
total_cells = weather["cell_id"].nunique()
batch_count = (total_cells + BATCH_CELL_COUNT - 1) // BATCH_CELL_COUNT

print("Total grid cells:", total_cells)
print("Cells per batch:", BATCH_CELL_COUNT)
print("Expected batches:", batch_count)
print("Expected rows per full batch:", BATCH_CELL_COUNT * 4383)
print("Compression:", PARQUET_COMPRESSION)
print("Output version:", FEATURE_DATASET_VERSION)
print("Output directory:", FEATURE_DATASET_DIR)

print("\nPyArrow version:", pa.__version__)
print("Parquet configuration: validated")


Total grid cells: 2520
Cells per batch: 10
Expected batches: 252
Expected rows per full batch: 43830
Compression: zstd
Output version: weather-features-v1
Output directory: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\data\processed\weather_features_v1\weather-features-v1

PyArrow version: 26.0.1
Parquet configuration: validated


In [19]:

# ============================================================
# WI-03 — CELL 11A
# Diagnose missing source-column error
# ============================================================

print("weather variable type:", type(weather))
print("weather shape:", getattr(weather, "shape", "Unavailable"))

print("\nweather columns:")
print(list(weather.columns))

print("\nRequired weather variables:")
print(WEATHER_VARIABLES)

print("\nMissing required columns:")
print([
    column for column in WEATHER_VARIABLES
    if column not in weather.columns
])

print("\nRaw dataset sample:")
display(weather.head(3))

print("\nExisting output files:")
for path in sorted(FEATURE_DATASET_DIR.glob("*")):
    if path.is_file():
        print(path.name, f"{path.stat().st_size / 1024**2:.2f} MiB")


weather variable type: <class 'pandas.core.frame.DataFrame'>
weather shape: (11045160, 9)

weather columns:
['cell_id', 'latitude', 'longitude', 'date', 'PRECTOTCORR', 'T2M_MAX', 'T2M_MIN', 'qc_pass', 'year']

Required weather variables:
['PRECTOTCORR', 'T2M_MAX', 'T2M_MIN']

Missing required columns:
[]

Raw dataset sample:


,cell_id,latitude,longitude,date,PRECTOTCORR,T2M_MAX,T2M_MIN,qc_pass,year
0,cell_0000,7.0,69.375,2013-01-01,1.63,28.39,27.01,True,2013
1,cell_0000,7.0,69.375,2013-01-02,0.63,27.45,27.06,True,2013
2,cell_0000,7.0,69.375,2013-01-03,0.27,27.81,27.08,True,2013



Existing output files:


In [20]:

# ============================================================
# WI-03 — CELL 11B
# Isolate batch selection and source columns
# ============================================================

batch_cells = all_cells[:BATCH_CELL_COUNT]

batch = (
    weather.loc[weather["cell_id"].isin(batch_cells)]
    .sort_values(["cell_id", "date"])
    .copy()
    .reset_index(drop=True)
)

print("Selected cells:", batch_cells)
print("Batch shape:", batch.shape)
print("Batch columns:", list(batch.columns))

assert all(
    variable in batch.columns
    for variable in WEATHER_VARIABLES
)

assert len(batch) == BATCH_CELL_COUNT * 4383

print("\nBatch source selection: PASSED")

# Test grouped access independently.
for variable in WEATHER_VARIABLES:
    grouped = batch.groupby("cell_id", sort=False)[variable]
    print(variable, "accessible:", len(grouped))
    
print("\nGrouped weather-variable access: PASSED")


Selected cells: ['cell_0000', 'cell_0001', 'cell_0002', 'cell_0003', 'cell_0004', 'cell_0005', 'cell_0006', 'cell_0007', 'cell_0008', 'cell_0009']
Batch shape: (43830, 9)
Batch columns: ['cell_id', 'latitude', 'longitude', 'date', 'PRECTOTCORR', 'T2M_MAX', 'T2M_MIN', 'qc_pass', 'year']

Batch source selection: PASSED
PRECTOTCORR accessible: 10
T2M_MAX accessible: 10
T2M_MIN accessible: 10

Grouped weather-variable access: PASSED


In [21]:

# ============================================================
# WI-03 — CELL 11C
# Generate first batch features, without writing to disk
# ============================================================

batch_test = batch.copy()

# Current-day predictors
for variable in WEATHER_VARIABLES:
    batch_test[f"{variable}_current"] = batch_test[variable]

# Calendar predictors
batch_test["day_of_year"] = batch_test["date"].dt.dayofyear
batch_test["month"] = batch_test["date"].dt.month
batch_test["day_of_week"] = batch_test["date"].dt.dayofweek

batch_test["day_of_year_sin"] = np.sin(
    2 * np.pi * batch_test["day_of_year"] / 365.25
)
batch_test["day_of_year_cos"] = np.cos(
    2 * np.pi * batch_test["day_of_year"] / 365.25
)
batch_test["day_of_week_sin"] = np.sin(
    2 * np.pi * batch_test["day_of_week"] / 7
)

# Lag and rolling predictors
for variable in WEATHER_VARIABLES:
    grouped = batch_test.groupby("cell_id", sort=False)[variable]

    for lag in LAG_DAYS:
        batch_test[f"{variable}_lag_{lag}"] = grouped.shift(lag)

    for window in ROLLING_WINDOWS:
        batch_test[f"{variable}_rolling_mean_{window}"] = (
            grouped.transform(
                lambda s: s.rolling(window, min_periods=window).mean()
            )
        )

        batch_test[f"{variable}_rolling_std_{window}"] = (
            grouped.transform(
                lambda s: s.rolling(window, min_periods=window).std()
            )
        )

# Forecast targets
for target in TARGET_VARIABLES:
    grouped = batch_test.groupby("cell_id", sort=False)[target]

    for horizon in FORECAST_HORIZONS:
        batch_test[f"{target}_target_t_plus_{horizon}"] = (
            grouped.shift(-horizon)
        )

# Validate exact schema
batch_test = batch_test[EXPECTED_COLUMNS].copy()

assert batch_test.columns.tolist() == EXPECTED_COLUMNS
assert len(batch_test) == 43830
assert not batch_test.duplicated(["cell_id", "date"]).any()

print("Feature generation: PASSED")
print("Batch shape:", batch_test.shape)
print(
    "Batch memory:",
    f"{batch_test.memory_usage(deep=True).sum() / 1024**2:.2f} MiB"
)
print("Missing feature rows:", int(
    batch_test[FEATURE_COLUMNS].isna().any(axis=1).sum()
))
print("Missing target rows:", int(
    batch_test[TARGET_COLUMNS].isna().any(axis=1).sum()
))
print("Canonical schema: PASSED")


Feature generation: PASSED
Batch shape: (43830, 69)
Batch memory: 24.66 MiB
Missing feature rows: 300
Missing target rows: 70
Canonical schema: PASSED


In [22]:

# ============================================================
# WI-03 — CELL 11D
# Write and verify first Parquet batch
# ============================================================

import time
import pyarrow as pa
import pyarrow.parquet as pq

batch_path = FEATURE_DATASET_DIR / "batch_0000.parquet"

# Never overwrite an existing batch.
if batch_path.exists():
    raise FileExistsError(
        f"Batch already exists: {batch_path}"
    )

table = pa.Table.from_pandas(
    batch_test,
    preserve_index=False
)

start_time = time.time()

pq.write_table(
    table,
    batch_path,
    compression=PARQUET_COMPRESSION,
    compression_level=PARQUET_COMPRESSION_LEVEL
)

write_seconds = time.time() - start_time

# Read back and verify persisted content.
check_table = pq.read_table(batch_path)

assert check_table.num_rows == len(batch_test)
assert check_table.column_names == EXPECTED_COLUMNS

check_df = check_table.to_pandas()

assert check_df["cell_id"].nunique() == BATCH_CELL_COUNT
assert not check_df.duplicated(["cell_id", "date"]).any()

file_size_mb = batch_path.stat().st_size / (1024 ** 2)

print("Parquet write: PASSED")
print("Rows:", f"{check_table.num_rows:,}")
print("Columns:", check_table.num_columns)
print("Grid cells:", check_df["cell_id"].nunique())
print(f"File size: {file_size_mb:.2f} MiB")
print(f"Write time: {write_seconds:.2f} seconds")
print("Read-back schema and key validation: PASSED")
print("Saved to:", batch_path)


Parquet write: PASSED
Rows: 43,830
Columns: 69
Grid cells: 10
File size: 7.97 MiB
Write time: 0.55 seconds
Read-back schema and key validation: PASSED
Saved to: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\data\processed\weather_features_v1\weather-features-v1\batch_0000.parquet


In [25]:

# ============================================================
# WI-03 — CELL 12A
# Verify and register the existing first batch
# ============================================================

import json
import pyarrow.parquet as pq

manifest_path = FEATURE_DATASET_DIR / "batch_manifest.json"
first_batch_path = FEATURE_DATASET_DIR / "batch_0000.parquet"

assert first_batch_path.exists(), "First batch file not found."

existing = pq.read_table(first_batch_path)
existing_df = existing.to_pandas()

expected_first_cells = all_cells[:BATCH_CELL_COUNT]

assert existing.column_names == EXPECTED_COLUMNS
assert existing.num_rows == BATCH_CELL_COUNT * 4383
assert existing_df["cell_id"].nunique() == BATCH_CELL_COUNT
assert sorted(existing_df["cell_id"].unique()) == expected_first_cells
assert not existing_df.duplicated(["cell_id", "date"]).any()

# Initialize or load manifest.
if manifest_path.exists():
    with open(manifest_path, "r", encoding="utf-8") as f:
        manifest = json.load(f)
else:
    manifest = {
        "dataset_version": FEATURE_DATASET_VERSION,
        "schema_columns": EXPECTED_COLUMNS,
        "batch_cell_count": BATCH_CELL_COUNT,
        "completed_batches": []
    }

# Confirm manifest compatibility.
assert manifest["dataset_version"] == FEATURE_DATASET_VERSION
assert manifest["schema_columns"] == EXPECTED_COLUMNS
assert manifest["batch_cell_count"] == BATCH_CELL_COUNT

# Register only if not already registered.
if not any(
    item["batch_index"] == 0
    for item in manifest["completed_batches"]
):
    manifest["completed_batches"].append({
        "batch_index": 0,
        "cell_start": expected_first_cells[0],
        "cell_end": expected_first_cells[-1],
        "cell_count": BATCH_CELL_COUNT,
        "row_count": existing.num_rows,
        "column_count": existing.num_columns,
        "file_name": first_batch_path.name,
        "file_size_mb": round(
            first_batch_path.stat().st_size / (1024 ** 2), 3
        ),
        "elapsed_seconds": 0.55,
        "status": "validated"
    })

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("Existing batch verified and registered.")
print("Batch index: 0")
print("Rows:", f"{existing.num_rows:,}")
print("Grid cells:", existing_df["cell_id"].nunique())
print("Schema validation: PASSED")
print("Manifest:", manifest_path)

del existing, existing_df


Existing batch verified and registered.
Batch index: 0
Rows: 43,830
Grid cells: 10
Schema validation: PASSED
Manifest: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\data\processed\weather_features_v1\weather-features-v1\batch_manifest.json


In [26]:

# ============================================================
# WI-03 — CELL 12B
# Safely process batches 0001–0005
# ============================================================

import gc
import json
import time
import pyarrow as pa
import pyarrow.parquet as pq

manifest_path = FEATURE_DATASET_DIR / "batch_manifest.json"

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

assert manifest["schema_columns"] == EXPECTED_COLUMNS

completed_indices = {
    item["batch_index"]
    for item in manifest["completed_batches"]
}

NEW_BATCH_INDICES = range(1, 6)

for batch_index in NEW_BATCH_INDICES:

    if batch_index in completed_indices:
        print(f"Batch {batch_index:04d}: already registered, skipping.")
        continue

    start = batch_index * BATCH_CELL_COUNT
    cells = all_cells[start:start + BATCH_CELL_COUNT]

    batch_path = FEATURE_DATASET_DIR / f"batch_{batch_index:04d}.parquet"

    if batch_path.exists():
        raise FileExistsError(
            f"Unregistered file exists; inspect before continuing: {batch_path}"
        )

    start_time = time.time()

    frame = build_feature_batch(cells)

    expected_rows = len(cells) * 4383

    assert len(frame) == expected_rows
    assert frame.columns.tolist() == EXPECTED_COLUMNS
    assert not frame.duplicated(["cell_id", "date"]).any()

    table = pa.Table.from_pandas(frame, preserve_index=False)

    pq.write_table(
        table,
        batch_path,
        compression=PARQUET_COMPRESSION,
        compression_level=PARQUET_COMPRESSION_LEVEL
    )

    check = pq.read_table(batch_path)

    assert check.num_rows == expected_rows
    assert check.column_names == EXPECTED_COLUMNS

    check_df = check.to_pandas()

    assert check_df["cell_id"].nunique() == len(cells)
    assert sorted(check_df["cell_id"].unique()) == cells

    elapsed = time.time() - start_time
    size_mb = batch_path.stat().st_size / (1024 ** 2)

    result = {
        "batch_index": batch_index,
        "cell_start": cells[0],
        "cell_end": cells[-1],
        "cell_count": len(cells),
        "row_count": expected_rows,
        "column_count": len(EXPECTED_COLUMNS),
        "file_name": batch_path.name,
        "file_size_mb": round(size_mb, 3),
        "elapsed_seconds": round(elapsed, 3),
        "status": "validated"
    }

    manifest["completed_batches"].append(result)

    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    completed_indices.add(batch_index)

    print(
        f"Batch {batch_index:04d} PASSED | "
        f"cells={len(cells)} | rows={expected_rows:,} | "
        f"size={size_mb:.2f} MiB | time={elapsed:.2f}s"
    )

    del frame, table, check, check_df
    gc.collect()

print("\nFive-batch test completed.")
print("Registered batches:", len(manifest["completed_batches"]))
print("Manifest:", manifest_path)


Batch 0001 PASSED | cells=10 | rows=43,830 | size=8.36 MiB | time=2.14s
Batch 0002 PASSED | cells=10 | rows=43,830 | size=8.32 MiB | time=2.12s
Batch 0003 PASSED | cells=10 | rows=43,830 | size=8.10 MiB | time=1.99s
Batch 0004 PASSED | cells=10 | rows=43,830 | size=8.03 MiB | time=1.91s
Batch 0005 PASSED | cells=10 | rows=43,830 | size=8.05 MiB | time=1.84s

Five-batch test completed.
Registered batches: 6
Manifest: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\data\processed\weather_features_v1\weather-features-v1\batch_manifest.json


In [27]:

# ============================================================
# WI-03 — CELL 13
# Audit currently generated feature batches
# ============================================================

import json
import pyarrow.parquet as pq
import pandas as pd

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

registered = sorted(
    manifest["completed_batches"],
    key=lambda item: item["batch_index"]
)

audit_rows = []
observed_cells = set()
total_rows = 0
total_size = 0

for item in registered:
    batch_path = FEATURE_DATASET_DIR / item["file_name"]

    assert batch_path.exists(), f"Missing file: {batch_path}"

    table = pq.read_table(batch_path)

    assert table.column_names == EXPECTED_COLUMNS
    assert table.num_rows == item["row_count"]

    frame = table.select(["cell_id", "date"]).to_pandas()

    assert not frame.duplicated(["cell_id", "date"]).any()

    cells = set(frame["cell_id"].unique())

    assert len(cells) == item["cell_count"]
    assert not observed_cells.intersection(cells), (
        f"Duplicate cell coverage in {item['file_name']}"
    )

    observed_cells.update(cells)
    total_rows += table.num_rows
    total_size += batch_path.stat().st_size

    audit_rows.append({
        "batch": item["batch_index"],
        "rows": table.num_rows,
        "cells": len(cells),
        "file_mb": round(batch_path.stat().st_size / 1024**2, 2),
        "status": "PASSED"
    })

audit_df = pd.DataFrame(audit_rows)

expected_cells = set(all_cells[:len(observed_cells)])

assert observed_cells == expected_cells
assert total_rows == len(observed_cells) * 4383

print(audit_df.to_string(index=False))
print("\nTotal registered batches:", len(registered))
print("Total grid cells:", len(observed_cells))
print("Total observations:", f"{total_rows:,}")
print(f"Total Parquet size: {total_size / 1024**2:.2f} MiB")
print("Cell coverage and duplicate-key audit: PASSED")


 batch  rows  cells  file_mb status
     0 43830     10     7.97 PASSED
     1 43830     10     8.36 PASSED
     2 43830     10     8.32 PASSED
     3 43830     10     8.10 PASSED
     4 43830     10     8.03 PASSED
     5 43830     10     8.05 PASSED

Total registered batches: 6
Total grid cells: 60
Total observations: 262,980
Total Parquet size: 48.82 MiB
Cell coverage and duplicate-key audit: PASSED


In [28]:

# ============================================================
# WI-03 — CELL 14
# Process batches 0006–0025 with checkpoints
# ============================================================

import gc
import json
import time
import shutil
import pyarrow as pa
import pyarrow.parquet as pq

START_BATCH = 6
END_BATCH = 26  # exclusive
MIN_FREE_GB = 3.0

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

completed_indices = {
    item["batch_index"]
    for item in manifest["completed_batches"]
}

for batch_index in range(START_BATCH, END_BATCH):

    if batch_index in completed_indices:
        print(f"Batch {batch_index:04d}: already completed, skipping.")
        continue

    disk_free_gb = shutil.disk_usage(PROJECT_ROOT).free / (1024 ** 3)

    if disk_free_gb < MIN_FREE_GB:
        raise RuntimeError(
            f"Stopping: only {disk_free_gb:.2f} GiB free."
        )

    start = batch_index * BATCH_CELL_COUNT
    cells = all_cells[start:start + BATCH_CELL_COUNT]

    batch_path = FEATURE_DATASET_DIR / f"batch_{batch_index:04d}.parquet"

    if batch_path.exists():
        raise FileExistsError(
            f"Unregistered file exists: {batch_path}"
        )

    start_time = time.time()

    frame = build_feature_batch(cells)

    expected_rows = len(cells) * 4383

    assert len(frame) == expected_rows
    assert frame.columns.tolist() == EXPECTED_COLUMNS
    assert not frame.duplicated(["cell_id", "date"]).any()

    table = pa.Table.from_pandas(frame, preserve_index=False)

    pq.write_table(
        table,
        batch_path,
        compression=PARQUET_COMPRESSION,
        compression_level=PARQUET_COMPRESSION_LEVEL
    )

    check = pq.read_table(batch_path)

    assert check.num_rows == expected_rows
    assert check.column_names == EXPECTED_COLUMNS

    check_cells = sorted(check.column("cell_id").to_pylist())
    assert sorted(set(check_cells)) == cells
    assert len(check_cells) == expected_rows

    elapsed = time.time() - start_time
    size_mb = batch_path.stat().st_size / (1024 ** 2)

    result = {
        "batch_index": batch_index,
        "cell_start": cells[0],
        "cell_end": cells[-1],
        "cell_count": len(cells),
        "row_count": expected_rows,
        "column_count": len(EXPECTED_COLUMNS),
        "file_name": batch_path.name,
        "file_size_mb": round(size_mb, 3),
        "elapsed_seconds": round(elapsed, 3),
        "status": "validated"
    }

    manifest["completed_batches"].append(result)

    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    completed_indices.add(batch_index)

    print(
        f"Batch {batch_index:04d} PASSED | "
        f"{cells[0]}–{cells[-1]} | "
        f"{size_mb:.2f} MiB | "
        f"{elapsed:.2f}s | "
        f"Free disk: {disk_free_gb:.2f} GiB"
    )

    del frame, table, check, check_cells
    gc.collect()

print("\nProcessing block completed.")
print("Registered batches:", len(manifest["completed_batches"]))
print("New batches processed:", END_BATCH - START_BATCH)


Batch 0006 PASSED | cell_0060–cell_0069 | 8.56 MiB | 1.91s | Free disk: 10.01 GiB
Batch 0007 PASSED | cell_0070–cell_0079 | 8.09 MiB | 2.00s | Free disk: 10.00 GiB
Batch 0008 PASSED | cell_0080–cell_0089 | 8.10 MiB | 1.89s | Free disk: 9.99 GiB
Batch 0009 PASSED | cell_0090–cell_0099 | 7.97 MiB | 1.99s | Free disk: 9.99 GiB
Batch 0010 PASSED | cell_0100–cell_0109 | 8.37 MiB | 1.99s | Free disk: 9.98 GiB
Batch 0011 PASSED | cell_0110–cell_0119 | 8.24 MiB | 2.05s | Free disk: 9.97 GiB
Batch 0012 PASSED | cell_0120–cell_0129 | 8.07 MiB | 2.04s | Free disk: 9.96 GiB
Batch 0013 PASSED | cell_0130–cell_0139 | 8.00 MiB | 2.00s | Free disk: 9.95 GiB
Batch 0014 PASSED | cell_0140–cell_0149 | 8.30 MiB | 2.08s | Free disk: 9.95 GiB
Batch 0015 PASSED | cell_0150–cell_0159 | 8.43 MiB | 1.99s | Free disk: 9.94 GiB
Batch 0016 PASSED | cell_0160–cell_0169 | 8.06 MiB | 2.04s | Free disk: 9.93 GiB
Batch 0017 PASSED | cell_0170–cell_0179 | 8.07 MiB | 2.01s | Free disk: 9.92 GiB
Batch 0018 PASSED | cell_0

In [29]:

# ============================================================
# WI-03 — CELL 15
# Audit batches 0000–0025
# ============================================================

import json
import pyarrow.parquet as pq
import pandas as pd

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

registered = sorted(
    manifest["completed_batches"],
    key=lambda item: item["batch_index"]
)

assert len(registered) == 26
assert [x["batch_index"] for x in registered] == list(range(26))

observed_cells = set()
total_rows = 0
total_size = 0

for item in registered:
    path = FEATURE_DATASET_DIR / item["file_name"]

    assert path.exists(), f"Missing file: {path}"

    table = pq.read_table(path)

    assert table.column_names == EXPECTED_COLUMNS
    assert table.num_rows == item["row_count"]

    keys = table.select(["cell_id", "date"]).to_pandas()

    assert not keys.duplicated(["cell_id", "date"]).any()

    cells = set(keys["cell_id"].unique())

    assert len(cells) == item["cell_count"]
    assert not observed_cells.intersection(cells)

    observed_cells.update(cells)
    total_rows += table.num_rows
    total_size += path.stat().st_size

expected_cells = set(all_cells[:260])

assert observed_cells == expected_cells
assert total_rows == 260 * 4383

print("Audited batches:", len(registered))
print("Grid cells:", len(observed_cells))
print("Observations:", f"{total_rows:,}")
print(f"Parquet storage: {total_size / 1024**2:.2f} MiB")
print("Missing batch files: 0")
print("Duplicate cell/date keys: 0")
print("Schema inconsistencies: 0")
print("Cell coverage audit: PASSED")


Audited batches: 26
Grid cells: 260
Observations: 1,139,580
Parquet storage: 212.79 MiB
Missing batch files: 0
Duplicate cell/date keys: 0
Schema inconsistencies: 0
Cell coverage audit: PASSED


In [30]:

# ============================================================
# WI-03 — CELL 16
# Process batches 0026–0045
# ============================================================

import gc
import json
import time
import shutil
import pyarrow as pa
import pyarrow.parquet as pq

START_BATCH = 26
END_BATCH = 46
MIN_FREE_GB = 3.0

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

completed_indices = {
    item["batch_index"]
    for item in manifest["completed_batches"]
}

for batch_index in range(START_BATCH, END_BATCH):

    if batch_index in completed_indices:
        print(f"Batch {batch_index:04d}: already completed, skipping.")
        continue

    free_gb = shutil.disk_usage(PROJECT_ROOT).free / (1024 ** 3)
    if free_gb < MIN_FREE_GB:
        raise RuntimeError(f"Low disk space: {free_gb:.2f} GiB")

    start = batch_index * BATCH_CELL_COUNT
    cells = all_cells[start:start + BATCH_CELL_COUNT]

    batch_path = FEATURE_DATASET_DIR / f"batch_{batch_index:04d}.parquet"

    if batch_path.exists():
        raise FileExistsError(f"Unregistered file exists: {batch_path}")

    start_time = time.time()
    frame = build_feature_batch(cells)

    expected_rows = len(cells) * 4383

    assert len(frame) == expected_rows
    assert frame.columns.tolist() == EXPECTED_COLUMNS
    assert not frame.duplicated(["cell_id", "date"]).any()

    table = pa.Table.from_pandas(frame, preserve_index=False)

    pq.write_table(
        table,
        batch_path,
        compression=PARQUET_COMPRESSION,
        compression_level=PARQUET_COMPRESSION_LEVEL
    )

    check = pq.read_table(batch_path)

    assert check.num_rows == expected_rows
    assert check.column_names == EXPECTED_COLUMNS
    assert sorted(set(check.column("cell_id").to_pylist())) == cells

    elapsed = time.time() - start_time
    size_mb = batch_path.stat().st_size / (1024 ** 2)

    manifest["completed_batches"].append({
        "batch_index": batch_index,
        "cell_start": cells[0],
        "cell_end": cells[-1],
        "cell_count": len(cells),
        "row_count": expected_rows,
        "column_count": len(EXPECTED_COLUMNS),
        "file_name": batch_path.name,
        "file_size_mb": round(size_mb, 3),
        "elapsed_seconds": round(elapsed, 3),
        "status": "validated"
    })

    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    print(
        f"Batch {batch_index:04d} PASSED | "
        f"{cells[0]}–{cells[-1]} | "
        f"{size_mb:.2f} MiB | "
        f"{elapsed:.2f}s"
    )

    completed_indices.add(batch_index)
    del frame, table, check
    gc.collect()

print("\nBlock completed.")
print("Registered batches:", len(manifest["completed_batches"]))


Batch 0026 PASSED | cell_0260–cell_0269 | 8.10 MiB | 0.84s
Batch 0027 PASSED | cell_0270–cell_0279 | 7.97 MiB | 0.95s
Batch 0028 PASSED | cell_0280–cell_0289 | 8.65 MiB | 1.23s
Batch 0029 PASSED | cell_0290–cell_0299 | 8.15 MiB | 1.15s
Batch 0030 PASSED | cell_0300–cell_0309 | 8.15 MiB | 1.09s
Batch 0031 PASSED | cell_0310–cell_0319 | 8.06 MiB | 1.13s
Batch 0032 PASSED | cell_0320–cell_0329 | 8.45 MiB | 1.11s
Batch 0033 PASSED | cell_0330–cell_0339 | 8.52 MiB | 1.09s
Batch 0034 PASSED | cell_0340–cell_0349 | 8.11 MiB | 1.08s
Batch 0035 PASSED | cell_0350–cell_0359 | 8.17 MiB | 1.03s
Batch 0036 PASSED | cell_0360–cell_0369 | 8.00 MiB | 1.12s
Batch 0037 PASSED | cell_0370–cell_0379 | 8.58 MiB | 1.11s
Batch 0038 PASSED | cell_0380–cell_0389 | 8.12 MiB | 1.18s
Batch 0039 PASSED | cell_0390–cell_0399 | 8.09 MiB | 1.16s
Batch 0040 PASSED | cell_0400–cell_0409 | 8.03 MiB | 1.17s
Batch 0041 PASSED | cell_0410–cell_0419 | 8.49 MiB | 1.06s
Batch 0042 PASSED | cell_0420–cell_0429 | 8.47 MiB | 1.4

In [31]:

# ============================================================
# WI-03 — CELL 17
# Audit all 46 registered batches
# ============================================================

import json
import pyarrow.parquet as pq

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

registered = sorted(
    manifest["completed_batches"],
    key=lambda item: item["batch_index"]
)

assert len(registered) == 46
assert [x["batch_index"] for x in registered] == list(range(46))

observed_cells = set()
total_rows = 0
total_size = 0

for item in registered:
    path = FEATURE_DATASET_DIR / item["file_name"]

    assert path.exists(), f"Missing file: {path}"

    table = pq.read_table(path)

    assert table.column_names == EXPECTED_COLUMNS
    assert table.num_rows == item["row_count"]

    keys = table.select(["cell_id", "date"]).to_pandas()

    assert not keys.duplicated(["cell_id", "date"]).any()

    cells = set(keys["cell_id"].unique())

    assert len(cells) == item["cell_count"]
    assert not observed_cells.intersection(cells)

    observed_cells.update(cells)
    total_rows += table.num_rows
    total_size += path.stat().st_size

expected_cells = set(all_cells[:460])

assert observed_cells == expected_cells
assert total_rows == 460 * 4383

print("Audited batches:", len(registered))
print("Grid cells:", len(observed_cells))
print("Observations:", f"{total_rows:,}")
print(f"Parquet storage: {total_size / 1024**2:.2f} MiB")
print("Missing files: 0")
print("Duplicate cell/date keys: 0")
print("Schema inconsistencies: 0")
print("Cell coverage audit: PASSED")


Audited batches: 46
Grid cells: 460
Observations: 2,016,180
Parquet storage: 377.11 MiB
Missing files: 0
Duplicate cell/date keys: 0
Schema inconsistencies: 0
Cell coverage audit: PASSED


In [32]:

# ============================================================
# WI-03 — CELL 18
# Process batches 0046–0065
# ============================================================

import gc
import json
import time
import shutil
import pyarrow as pa
import pyarrow.parquet as pq

START_BATCH = 46
END_BATCH = 66
MIN_FREE_GB = 3.0

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

completed_indices = {
    item["batch_index"]
    for item in manifest["completed_batches"]
}

for batch_index in range(START_BATCH, END_BATCH):

    if batch_index in completed_indices:
        print(f"Batch {batch_index:04d}: already completed, skipping.")
        continue

    free_gb = shutil.disk_usage(PROJECT_ROOT).free / (1024 ** 3)
    if free_gb < MIN_FREE_GB:
        raise RuntimeError(f"Low disk space: {free_gb:.2f} GiB")

    start = batch_index * BATCH_CELL_COUNT
    cells = all_cells[start:start + BATCH_CELL_COUNT]

    batch_path = FEATURE_DATASET_DIR / f"batch_{batch_index:04d}.parquet"

    if batch_path.exists():
        raise FileExistsError(f"Unregistered file exists: {batch_path}")

    start_time = time.time()
    frame = build_feature_batch(cells)

    expected_rows = len(cells) * 4383

    assert len(frame) == expected_rows
    assert frame.columns.tolist() == EXPECTED_COLUMNS
    assert not frame.duplicated(["cell_id", "date"]).any()

    table = pa.Table.from_pandas(frame, preserve_index=False)

    pq.write_table(
        table,
        batch_path,
        compression=PARQUET_COMPRESSION,
        compression_level=PARQUET_COMPRESSION_LEVEL
    )

    check = pq.read_table(batch_path)

    assert check.num_rows == expected_rows
    assert check.column_names == EXPECTED_COLUMNS
    assert sorted(set(check.column("cell_id").to_pylist())) == cells

    elapsed = time.time() - start_time
    size_mb = batch_path.stat().st_size / (1024 ** 2)

    manifest["completed_batches"].append({
        "batch_index": batch_index,
        "cell_start": cells[0],
        "cell_end": cells[-1],
        "cell_count": len(cells),
        "row_count": expected_rows,
        "column_count": len(EXPECTED_COLUMNS),
        "file_name": batch_path.name,
        "file_size_mb": round(size_mb, 3),
        "elapsed_seconds": round(elapsed, 3),
        "status": "validated"
    })

    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    print(
        f"Batch {batch_index:04d} PASSED | "
        f"{cells[0]}–{cells[-1]} | "
        f"{size_mb:.2f} MiB | "
        f"{elapsed:.2f}s | "
        f"Free disk: {free_gb:.2f} GiB"
    )

    completed_indices.add(batch_index)
    del frame, table, check
    gc.collect()

print("\nBlock completed.")
print("Registered batches:", len(manifest["completed_batches"]))


Batch 0046 PASSED | cell_0460–cell_0469 | 8.56 MiB | 0.95s | Free disk: 9.76 GiB
Batch 0047 PASSED | cell_0470–cell_0479 | 8.07 MiB | 1.14s | Free disk: 9.75 GiB
Batch 0048 PASSED | cell_0480–cell_0489 | 8.10 MiB | 1.12s | Free disk: 9.74 GiB
Batch 0049 PASSED | cell_0490–cell_0499 | 7.97 MiB | 0.96s | Free disk: 9.73 GiB
Batch 0050 PASSED | cell_0500–cell_0509 | 8.50 MiB | 1.10s | Free disk: 9.73 GiB
Batch 0051 PASSED | cell_0510–cell_0519 | 8.43 MiB | 1.02s | Free disk: 9.72 GiB
Batch 0052 PASSED | cell_0520–cell_0529 | 8.01 MiB | 1.04s | Free disk: 9.71 GiB
Batch 0053 PASSED | cell_0530–cell_0539 | 8.15 MiB | 1.01s | Free disk: 9.70 GiB
Batch 0054 PASSED | cell_0540–cell_0549 | 8.07 MiB | 1.00s | Free disk: 9.69 GiB
Batch 0055 PASSED | cell_0550–cell_0559 | 8.36 MiB | 1.08s | Free disk: 9.69 GiB
Batch 0056 PASSED | cell_0560–cell_0569 | 8.00 MiB | 0.98s | Free disk: 9.68 GiB
Batch 0057 PASSED | cell_0570–cell_0579 | 8.09 MiB | 1.09s | Free disk: 9.67 GiB
Batch 0058 PASSED | cell_058

In [33]:

# ============================================================
# WI-03 — CELL 19
# Audit all 66 registered batches
# ============================================================

import json
import pyarrow.parquet as pq

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

registered = sorted(
    manifest["completed_batches"],
    key=lambda item: item["batch_index"]
)

assert len(registered) == 66
assert [x["batch_index"] for x in registered] == list(range(66))

observed_cells = set()
total_rows = 0
total_size = 0

for item in registered:
    path = FEATURE_DATASET_DIR / item["file_name"]

    assert path.exists(), f"Missing file: {path}"

    table = pq.read_table(path)

    assert table.column_names == EXPECTED_COLUMNS
    assert table.num_rows == item["row_count"]

    keys = table.select(["cell_id", "date"]).to_pandas()

    assert not keys.duplicated(["cell_id", "date"]).any()

    cells = set(keys["cell_id"].unique())

    assert len(cells) == item["cell_count"]
    assert not observed_cells.intersection(cells)

    observed_cells.update(cells)
    total_rows += table.num_rows
    total_size += path.stat().st_size

assert observed_cells == set(all_cells[:660])
assert total_rows == 660 * 4383

print("Audited batches:", len(registered))
print("Grid cells:", len(observed_cells))
print("Observations:", f"{total_rows:,}")
print(f"Parquet storage: {total_size / 1024**2:.2f} MiB")
print("Missing files: 0")
print("Duplicate cell/date keys: 0")
print("Schema inconsistencies: 0")
print("Cell coverage audit: PASSED")


Audited batches: 66
Grid cells: 660
Observations: 2,892,780
Parquet storage: 539.83 MiB
Missing files: 0
Duplicate cell/date keys: 0
Schema inconsistencies: 0
Cell coverage audit: PASSED


In [34]:

# ============================================================
# WI-03 — CELL 20
# Process batches 0066–0085
# ============================================================

import gc
import json
import time
import shutil
import pyarrow as pa
import pyarrow.parquet as pq

START_BATCH = 66
END_BATCH = 86
MIN_FREE_GB = 3.0

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

completed_indices = {
    item["batch_index"]
    for item in manifest["completed_batches"]
}

for batch_index in range(START_BATCH, END_BATCH):

    if batch_index in completed_indices:
        print(f"Batch {batch_index:04d}: already completed, skipping.")
        continue

    free_gb = shutil.disk_usage(PROJECT_ROOT).free / (1024 ** 3)
    if free_gb < MIN_FREE_GB:
        raise RuntimeError(f"Low disk space: {free_gb:.2f} GiB")

    start = batch_index * BATCH_CELL_COUNT
    cells = all_cells[start:start + BATCH_CELL_COUNT]

    batch_path = FEATURE_DATASET_DIR / f"batch_{batch_index:04d}.parquet"

    if batch_path.exists():
        raise FileExistsError(f"Unregistered file exists: {batch_path}")

    start_time = time.time()
    frame = build_feature_batch(cells)

    expected_rows = len(cells) * 4383

    assert len(frame) == expected_rows
    assert frame.columns.tolist() == EXPECTED_COLUMNS
    assert not frame.duplicated(["cell_id", "date"]).any()

    table = pa.Table.from_pandas(frame, preserve_index=False)

    pq.write_table(
        table,
        batch_path,
        compression=PARQUET_COMPRESSION,
        compression_level=PARQUET_COMPRESSION_LEVEL
    )

    check = pq.read_table(batch_path)

    assert check.num_rows == expected_rows
    assert check.column_names == EXPECTED_COLUMNS
    assert sorted(set(check.column("cell_id").to_pylist())) == cells

    elapsed = time.time() - start_time
    size_mb = batch_path.stat().st_size / (1024 ** 2)

    manifest["completed_batches"].append({
        "batch_index": batch_index,
        "cell_start": cells[0],
        "cell_end": cells[-1],
        "cell_count": len(cells),
        "row_count": expected_rows,
        "column_count": len(EXPECTED_COLUMNS),
        "file_name": batch_path.name,
        "file_size_mb": round(size_mb, 3),
        "elapsed_seconds": round(elapsed, 3),
        "status": "validated"
    })

    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    print(
        f"Batch {batch_index:04d} PASSED | "
        f"{cells[0]}–{cells[-1]} | "
        f"{size_mb:.2f} MiB | "
        f"{elapsed:.2f}s"
    )

    completed_indices.add(batch_index)
    del frame, table, check
    gc.collect()

print("\nBlock completed.")
print("Registered batches:", len(manifest["completed_batches"]))


Batch 0066 PASSED | cell_0660–cell_0669 | 7.99 MiB | 0.95s
Batch 0067 PASSED | cell_0670–cell_0679 | 7.86 MiB | 0.98s
Batch 0068 PASSED | cell_0680–cell_0689 | 8.16 MiB | 1.00s
Batch 0069 PASSED | cell_0690–cell_0699 | 8.18 MiB | 0.96s
Batch 0070 PASSED | cell_0700–cell_0709 | 7.92 MiB | 0.96s
Batch 0071 PASSED | cell_0710–cell_0719 | 8.02 MiB | 1.04s
Batch 0072 PASSED | cell_0720–cell_0729 | 7.97 MiB | 1.04s
Batch 0073 PASSED | cell_0730–cell_0739 | 8.08 MiB | 1.16s
Batch 0074 PASSED | cell_0740–cell_0749 | 7.94 MiB | 1.20s
Batch 0075 PASSED | cell_0750–cell_0759 | 8.00 MiB | 1.06s
Batch 0076 PASSED | cell_0760–cell_0769 | 7.85 MiB | 1.07s
Batch 0077 PASSED | cell_0770–cell_0779 | 8.04 MiB | 1.08s
Batch 0078 PASSED | cell_0780–cell_0789 | 8.16 MiB | 1.05s
Batch 0079 PASSED | cell_0790–cell_0799 | 7.94 MiB | 1.07s
Batch 0080 PASSED | cell_0800–cell_0809 | 8.04 MiB | 1.03s
Batch 0081 PASSED | cell_0810–cell_0819 | 7.90 MiB | 1.06s
Batch 0082 PASSED | cell_0820–cell_0829 | 7.99 MiB | 1.1

In [35]:

# ============================================================
# WI-03 — CELL 21
# Audit all 86 registered batches
# ============================================================

import json
import pyarrow.parquet as pq

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

registered = sorted(
    manifest["completed_batches"],
    key=lambda item: item["batch_index"]
)

assert len(registered) == 86
assert [x["batch_index"] for x in registered] == list(range(86))

observed_cells = set()
total_rows = 0
total_size = 0

for item in registered:
    path = FEATURE_DATASET_DIR / item["file_name"]

    assert path.exists(), f"Missing file: {path}"

    table = pq.read_table(path)

    assert table.column_names == EXPECTED_COLUMNS
    assert table.num_rows == item["row_count"]

    keys = table.select(["cell_id", "date"]).to_pandas()
    assert not keys.duplicated(["cell_id", "date"]).any()

    cells = set(keys["cell_id"].unique())
    assert len(cells) == item["cell_count"]
    assert not observed_cells.intersection(cells)

    observed_cells.update(cells)
    total_rows += table.num_rows
    total_size += path.stat().st_size

assert observed_cells == set(all_cells[:860])
assert total_rows == 860 * 4383

print("Audited batches:", len(registered))
print("Grid cells:", len(observed_cells))
print("Observations:", f"{total_rows:,}")
print(f"Parquet storage: {total_size / 1024**2:.2f} MiB")
print("Missing files: 0")
print("Duplicate cell/date keys: 0")
print("Schema inconsistencies: 0")
print("Cell coverage audit: PASSED")


Audited batches: 86
Grid cells: 860
Observations: 3,769,380
Parquet storage: 699.75 MiB
Missing files: 0
Duplicate cell/date keys: 0
Schema inconsistencies: 0
Cell coverage audit: PASSED


In [36]:

# ============================================================
# WI-03 — CELL 22
# Process batches 0086–0105
# ============================================================

START_BATCH = 86
END_BATCH = 106  # exclusive
MIN_FREE_GB = 3.0

import gc
import json
import shutil
import pyarrow as pa
import pyarrow.parquet as pq

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

completed_indices = {
    item["batch_index"] for item in manifest["completed_batches"]
}

for batch_index in range(START_BATCH, END_BATCH):

    if batch_index in completed_indices:
        print(f"Batch {batch_index:04d} already registered — skipping.")
        continue

    free_gb = shutil.disk_usage(PROJECT_ROOT).free / (1024 ** 3)
    assert free_gb >= MIN_FREE_GB, (
        f"Free disk space below safety threshold: {free_gb:.2f} GiB"
    )

    cells = all_cells[
        batch_index * 10:(batch_index + 1) * 10
    ]

    assert len(cells) > 0, f"No cells for batch {batch_index}"

    batch_path = (
        FEATURE_DATASET_DIR /
        f"batch_{batch_index:04d}.parquet"
    )

    assert not batch_path.exists(), (
        f"Unregistered output already exists: {batch_path}. "
        "Inspect it before proceeding; do not overwrite."
    )

    frame = build_feature_batch(cells)

    expected_rows = len(cells) * 4383

    assert len(frame) == expected_rows
    assert frame.columns.tolist() == EXPECTED_COLUMNS
    assert not frame.duplicated(["cell_id", "date"]).any()

    table = pa.Table.from_pandas(frame, preserve_index=False)

    pq.write_table(
        table,
        batch_path,
        compression="zstd",
        compression_level=3
    )

    check = pq.read_table(batch_path)

    assert check.num_rows == expected_rows
    assert check.column_names == EXPECTED_COLUMNS
    assert set(check.column("cell_id").to_pylist()) == set(cells)

    manifest["completed_batches"].append({
        "batch_index": batch_index,
        "file_name": batch_path.name,
        "cell_count": len(cells),
        "row_count": expected_rows,
        "size_bytes": batch_path.stat().st_size
    })

    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    completed_indices.add(batch_index)

    print(
        f"Batch {batch_index:04d} PASSED | "
        f"{cells[0]}–{cells[-1]} | "
        f"{batch_path.stat().st_size / 1024**2:.2f} MiB"
    )

    del frame, table, check
    gc.collect()

print("\nBlock completed.")
print("Registered batches:", len(completed_indices))


Batch 0086 PASSED | cell_0860–cell_0869 | 8.00 MiB
Batch 0087 PASSED | cell_0870–cell_0879 | 8.26 MiB
Batch 0088 PASSED | cell_0880–cell_0889 | 7.91 MiB
Batch 0089 PASSED | cell_0890–cell_0899 | 8.20 MiB
Batch 0090 PASSED | cell_0900–cell_0909 | 7.87 MiB
Batch 0091 PASSED | cell_0910–cell_0919 | 8.00 MiB
Batch 0092 PASSED | cell_0920–cell_0929 | 8.25 MiB
Batch 0093 PASSED | cell_0930–cell_0939 | 7.95 MiB
Batch 0094 PASSED | cell_0940–cell_0949 | 8.03 MiB
Batch 0095 PASSED | cell_0950–cell_0959 | 8.03 MiB
Batch 0096 PASSED | cell_0960–cell_0969 | 8.33 MiB
Batch 0097 PASSED | cell_0970–cell_0979 | 7.94 MiB
Batch 0098 PASSED | cell_0980–cell_0989 | 8.23 MiB
Batch 0099 PASSED | cell_0990–cell_0999 | 7.95 MiB
Batch 0100 PASSED | cell_1000–cell_1009 | 8.08 MiB
Batch 0101 PASSED | cell_1010–cell_1019 | 8.28 MiB
Batch 0102 PASSED | cell_1020–cell_1029 | 8.06 MiB
Batch 0103 PASSED | cell_1030–cell_1039 | 8.00 MiB
Batch 0104 PASSED | cell_1040–cell_1049 | 8.10 MiB
Batch 0105 PASSED | cell_1050–c

In [37]:

# ============================================================
# WI-03 — CELL 23
# Audit all 106 registered batches
# ============================================================

import json
import pyarrow.parquet as pq

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

registered = sorted(
    manifest["completed_batches"],
    key=lambda item: item["batch_index"]
)

assert len(registered) == 106
assert [x["batch_index"] for x in registered] == list(range(106))

observed_cells = set()
total_rows = 0
total_size = 0

for item in registered:
    path = FEATURE_DATASET_DIR / item["file_name"]

    assert path.exists(), f"Missing file: {path}"

    table = pq.read_table(path)

    assert table.column_names == EXPECTED_COLUMNS
    assert table.num_rows == item["row_count"]

    keys = table.select(["cell_id", "date"]).to_pandas()
    assert not keys.duplicated(["cell_id", "date"]).any()

    cells = set(keys["cell_id"].unique())
    assert len(cells) == item["cell_count"]
    assert not observed_cells.intersection(cells)

    observed_cells.update(cells)
    total_rows += table.num_rows
    total_size += path.stat().st_size

assert observed_cells == set(all_cells[:1060])
assert total_rows == 1060 * 4383

print("Audited batches:", len(registered))
print("Grid cells:", len(observed_cells))
print("Observations:", f"{total_rows:,}")
print(f"Parquet storage: {total_size / 1024**2:.2f} MiB")
print("Missing files: 0")
print("Duplicate cell/date keys: 0")
print("Schema inconsistencies: 0")
print("Cell coverage audit: PASSED")


Audited batches: 106
Grid cells: 1060
Observations: 4,645,980
Parquet storage: 861.56 MiB
Missing files: 0
Duplicate cell/date keys: 0
Schema inconsistencies: 0
Cell coverage audit: PASSED


In [38]:

# ============================================================
# WI-03 — CELL 24
# Process batches 0106–0125
# ============================================================

START_BATCH = 106
END_BATCH = 126  # exclusive
MIN_FREE_GB = 3.0

import gc
import json
import shutil
import pyarrow as pa
import pyarrow.parquet as pq

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

completed_indices = {
    item["batch_index"] for item in manifest["completed_batches"]
}

for batch_index in range(START_BATCH, END_BATCH):

    if batch_index in completed_indices:
        print(f"Batch {batch_index:04d} already registered — skipping.")
        continue

    free_gb = shutil.disk_usage(PROJECT_ROOT).free / (1024 ** 3)
    assert free_gb >= MIN_FREE_GB, (
        f"Free disk space below safety threshold: {free_gb:.2f} GiB"
    )

    cells = all_cells[
        batch_index * 10:(batch_index + 1) * 10
    ]

    assert len(cells) > 0, f"No cells for batch {batch_index}"

    batch_path = (
        FEATURE_DATASET_DIR /
        f"batch_{batch_index:04d}.parquet"
    )

    assert not batch_path.exists(), (
        f"Unregistered output already exists: {batch_path}. "
        "Inspect it before proceeding; do not overwrite."
    )

    frame = build_feature_batch(cells)
    expected_rows = len(cells) * 4383

    assert len(frame) == expected_rows
    assert frame.columns.tolist() == EXPECTED_COLUMNS
    assert not frame.duplicated(["cell_id", "date"]).any()

    table = pa.Table.from_pandas(frame, preserve_index=False)

    pq.write_table(
        table,
        batch_path,
        compression="zstd",
        compression_level=3
    )

    check = pq.read_table(batch_path)

    assert check.num_rows == expected_rows
    assert check.column_names == EXPECTED_COLUMNS
    assert set(check.column("cell_id").to_pylist()) == set(cells)

    manifest["completed_batches"].append({
        "batch_index": batch_index,
        "file_name": batch_path.name,
        "cell_count": len(cells),
        "row_count": expected_rows,
        "size_bytes": batch_path.stat().st_size
    })

    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    completed_indices.add(batch_index)

    print(
        f"Batch {batch_index:04d} PASSED | "
        f"{cells[0]}–{cells[-1]} | "
        f"{batch_path.stat().st_size / 1024**2:.2f} MiB"
    )

    del frame, table, check
    gc.collect()

print("\nBlock completed.")
print("Registered batches:", len(completed_indices))


Batch 0106 PASSED | cell_1060–cell_1069 | 7.86 MiB
Batch 0107 PASSED | cell_1070–cell_1079 | 8.31 MiB
Batch 0108 PASSED | cell_1080–cell_1089 | 7.87 MiB
Batch 0109 PASSED | cell_1090–cell_1099 | 8.13 MiB
Batch 0110 PASSED | cell_1100–cell_1109 | 8.32 MiB
Batch 0111 PASSED | cell_1110–cell_1119 | 8.12 MiB
Batch 0112 PASSED | cell_1120–cell_1129 | 7.98 MiB
Batch 0113 PASSED | cell_1130–cell_1139 | 8.04 MiB
Batch 0114 PASSED | cell_1140–cell_1149 | 8.35 MiB
Batch 0115 PASSED | cell_1150–cell_1159 | 8.18 MiB
Batch 0116 PASSED | cell_1160–cell_1169 | 8.43 MiB
Batch 0117 PASSED | cell_1170–cell_1179 | 7.78 MiB
Batch 0118 PASSED | cell_1180–cell_1189 | 8.16 MiB
Batch 0119 PASSED | cell_1190–cell_1199 | 8.45 MiB
Batch 0120 PASSED | cell_1200–cell_1209 | 8.34 MiB
Batch 0121 PASSED | cell_1210–cell_1219 | 7.90 MiB
Batch 0122 PASSED | cell_1220–cell_1229 | 7.99 MiB
Batch 0123 PASSED | cell_1230–cell_1239 | 8.39 MiB
Batch 0124 PASSED | cell_1240–cell_1249 | 8.50 MiB
Batch 0125 PASSED | cell_1250–c

In [39]:

# ============================================================
# WI-03 — CELL 25
# Audit all 126 registered batches
# ============================================================

import json
import pyarrow.parquet as pq

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

registered = sorted(
    manifest["completed_batches"],
    key=lambda item: item["batch_index"]
)

assert len(registered) == 126
assert [x["batch_index"] for x in registered] == list(range(126))

observed_cells = set()
total_rows = 0
total_size = 0

for item in registered:
    path = FEATURE_DATASET_DIR / item["file_name"]

    assert path.exists(), f"Missing file: {path}"

    table = pq.read_table(path)

    assert table.column_names == EXPECTED_COLUMNS
    assert table.num_rows == item["row_count"]

    keys = table.select(["cell_id", "date"]).to_pandas()
    assert not keys.duplicated(["cell_id", "date"]).any()

    cells = set(keys["cell_id"].unique())
    assert len(cells) == item["cell_count"]
    assert not observed_cells.intersection(cells)

    observed_cells.update(cells)
    total_rows += table.num_rows
    total_size += path.stat().st_size

assert observed_cells == set(all_cells[:1260])
assert total_rows == 1260 * 4383

print("Audited batches:", len(registered))
print("Grid cells:", len(observed_cells))
print("Observations:", f"{total_rows:,}")
print(f"Parquet storage: {total_size / 1024**2:.2f} MiB")
print("Missing files: 0")
print("Duplicate cell/date keys: 0")
print("Schema inconsistencies: 0")
print("Cell coverage audit: PASSED")


Audited batches: 126
Grid cells: 1260
Observations: 5,522,580
Parquet storage: 1025.24 MiB
Missing files: 0
Duplicate cell/date keys: 0
Schema inconsistencies: 0
Cell coverage audit: PASSED


In [40]:

# ============================================================
# WI-03 — CELL 26
# Process batches 0126–0145
# ============================================================

START_BATCH = 126
END_BATCH = 146  # exclusive
MIN_FREE_GB = 3.0

import gc
import json
import shutil
import pyarrow as pa
import pyarrow.parquet as pq

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

completed_indices = {
    item["batch_index"] for item in manifest["completed_batches"]
}

for batch_index in range(START_BATCH, END_BATCH):

    if batch_index in completed_indices:
        print(f"Batch {batch_index:04d} already registered — skipping.")
        continue

    free_gb = shutil.disk_usage(PROJECT_ROOT).free / (1024 ** 3)
    assert free_gb >= MIN_FREE_GB, (
        f"Free disk space below safety threshold: {free_gb:.2f} GiB"
    )

    cells = all_cells[
        batch_index * 10:(batch_index + 1) * 10
    ]

    assert len(cells) > 0, f"No cells for batch {batch_index}"

    batch_path = (
        FEATURE_DATASET_DIR /
        f"batch_{batch_index:04d}.parquet"
    )

    assert not batch_path.exists(), (
        f"Unregistered output already exists: {batch_path}. "
        "Inspect it before proceeding; do not overwrite."
    )

    frame = build_feature_batch(cells)
    expected_rows = len(cells) * 4383

    assert len(frame) == expected_rows
    assert frame.columns.tolist() == EXPECTED_COLUMNS
    assert not frame.duplicated(["cell_id", "date"]).any()

    table = pa.Table.from_pandas(frame, preserve_index=False)

    pq.write_table(
        table,
        batch_path,
        compression="zstd",
        compression_level=3
    )

    check = pq.read_table(batch_path)

    assert check.num_rows == expected_rows
    assert check.column_names == EXPECTED_COLUMNS
    assert set(check.column("cell_id").to_pylist()) == set(cells)

    manifest["completed_batches"].append({
        "batch_index": batch_index,
        "file_name": batch_path.name,
        "cell_count": len(cells),
        "row_count": expected_rows,
        "size_bytes": batch_path.stat().st_size
    })

    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    completed_indices.add(batch_index)

    print(
        f"Batch {batch_index:04d} PASSED | "
        f"{cells[0]}–{cells[-1]} | "
        f"{batch_path.stat().st_size / 1024**2:.2f} MiB"
    )

    del frame, table, check
    gc.collect()

print("\nBlock completed.")
print("Registered batches:", len(completed_indices))


Batch 0126 PASSED | cell_1260–cell_1269 | 7.60 MiB
Batch 0127 PASSED | cell_1270–cell_1279 | 8.18 MiB
Batch 0128 PASSED | cell_1280–cell_1289 | 8.53 MiB
Batch 0129 PASSED | cell_1290–cell_1299 | 8.41 MiB
Batch 0130 PASSED | cell_1300–cell_1309 | 7.90 MiB
Batch 0131 PASSED | cell_1310–cell_1319 | 7.91 MiB
Batch 0132 PASSED | cell_1320–cell_1329 | 8.37 MiB
Batch 0133 PASSED | cell_1330–cell_1339 | 8.58 MiB
Batch 0134 PASSED | cell_1340–cell_1349 | 8.61 MiB
Batch 0135 PASSED | cell_1350–cell_1359 | 7.56 MiB
Batch 0136 PASSED | cell_1360–cell_1369 | 8.17 MiB
Batch 0137 PASSED | cell_1370–cell_1379 | 8.51 MiB
Batch 0138 PASSED | cell_1380–cell_1389 | 8.56 MiB
Batch 0139 PASSED | cell_1390–cell_1399 | 7.87 MiB
Batch 0140 PASSED | cell_1400–cell_1409 | 7.87 MiB
Batch 0141 PASSED | cell_1410–cell_1419 | 8.33 MiB
Batch 0142 PASSED | cell_1420–cell_1429 | 8.57 MiB
Batch 0143 PASSED | cell_1430–cell_1439 | 8.52 MiB
Batch 0144 PASSED | cell_1440–cell_1449 | 7.52 MiB
Batch 0145 PASSED | cell_1450–c

In [41]:

# ============================================================
# WI-03 — CELL 27
# Audit all 146 registered batches
# ============================================================

import json
import pyarrow.parquet as pq

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

registered = sorted(
    manifest["completed_batches"],
    key=lambda item: item["batch_index"]
)

assert len(registered) == 146
assert [x["batch_index"] for x in registered] == list(range(146))

observed_cells = set()
total_rows = 0
total_size = 0

for item in registered:
    path = FEATURE_DATASET_DIR / item["file_name"]

    assert path.exists(), f"Missing file: {path}"

    table = pq.read_table(path)

    assert table.column_names == EXPECTED_COLUMNS
    assert table.num_rows == item["row_count"]

    keys = table.select(["cell_id", "date"]).to_pandas()
    assert not keys.duplicated(["cell_id", "date"]).any()

    cells = set(keys["cell_id"].unique())
    assert len(cells) == item["cell_count"]
    assert not observed_cells.intersection(cells)

    observed_cells.update(cells)
    total_rows += table.num_rows
    total_size += path.stat().st_size

assert observed_cells == set(all_cells[:1460])
assert total_rows == 1460 * 4383

print("Audited batches:", len(registered))
print("Grid cells:", len(observed_cells))
print("Observations:", f"{total_rows:,}")
print(f"Parquet storage: {total_size / 1024**2:.2f} MiB")
print("Missing files: 0")
print("Duplicate cell/date keys: 0")
print("Schema inconsistencies: 0")
print("Cell coverage audit: PASSED")


Audited batches: 146
Grid cells: 1460
Observations: 6,399,180
Parquet storage: 1188.99 MiB
Missing files: 0
Duplicate cell/date keys: 0
Schema inconsistencies: 0
Cell coverage audit: PASSED


In [42]:

# ============================================================
# WI-03 — CELL 28
# Process batches 0146–0165
# ============================================================

START_BATCH = 146
END_BATCH = 166  # exclusive
MIN_FREE_GB = 3.0

import gc
import json
import shutil
import pyarrow as pa
import pyarrow.parquet as pq

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

completed_indices = {
    item["batch_index"] for item in manifest["completed_batches"]
}

for batch_index in range(START_BATCH, END_BATCH):

    if batch_index in completed_indices:
        print(f"Batch {batch_index:04d} already registered — skipping.")
        continue

    free_gb = shutil.disk_usage(PROJECT_ROOT).free / (1024 ** 3)
    assert free_gb >= MIN_FREE_GB, (
        f"Free disk space below safety threshold: {free_gb:.2f} GiB"
    )

    cells = all_cells[
        batch_index * 10:(batch_index + 1) * 10
    ]

    assert len(cells) > 0, f"No cells for batch {batch_index}"

    batch_path = (
        FEATURE_DATASET_DIR /
        f"batch_{batch_index:04d}.parquet"
    )

    assert not batch_path.exists(), (
        f"Unregistered output already exists: {batch_path}. "
        "Inspect it before proceeding; do not overwrite."
    )

    frame = build_feature_batch(cells)
    expected_rows = len(cells) * 4383

    assert len(frame) == expected_rows
    assert frame.columns.tolist() == EXPECTED_COLUMNS
    assert not frame.duplicated(["cell_id", "date"]).any()

    table = pa.Table.from_pandas(frame, preserve_index=False)

    pq.write_table(
        table,
        batch_path,
        compression="zstd",
        compression_level=3
    )

    check = pq.read_table(batch_path)

    assert check.num_rows == expected_rows
    assert check.column_names == EXPECTED_COLUMNS
    assert set(check.column("cell_id").to_pylist()) == set(cells)

    manifest["completed_batches"].append({
        "batch_index": batch_index,
        "file_name": batch_path.name,
        "cell_count": len(cells),
        "row_count": expected_rows,
        "size_bytes": batch_path.stat().st_size
    })

    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    completed_indices.add(batch_index)

    print(
        f"Batch {batch_index:04d} PASSED | "
        f"{cells[0]}–{cells[-1]} | "
        f"{batch_path.stat().st_size / 1024**2:.2f} MiB"
    )

    del frame, table, check
    gc.collect()

print("\nBlock completed.")
print("Registered batches:", len(completed_indices))


Batch 0146 PASSED | cell_1460–cell_1469 | 8.44 MiB
Batch 0147 PASSED | cell_1470–cell_1479 | 8.62 MiB
Batch 0148 PASSED | cell_1480–cell_1489 | 7.83 MiB
Batch 0149 PASSED | cell_1490–cell_1499 | 7.86 MiB
Batch 0150 PASSED | cell_1500–cell_1509 | 8.32 MiB
Batch 0151 PASSED | cell_1510–cell_1519 | 8.54 MiB
Batch 0152 PASSED | cell_1520–cell_1529 | 8.52 MiB
Batch 0153 PASSED | cell_1530–cell_1539 | 7.51 MiB
Batch 0154 PASSED | cell_1540–cell_1549 | 8.12 MiB
Batch 0155 PASSED | cell_1550–cell_1559 | 8.40 MiB
Batch 0156 PASSED | cell_1560–cell_1569 | 8.56 MiB
Batch 0157 PASSED | cell_1570–cell_1579 | 7.91 MiB
Batch 0158 PASSED | cell_1580–cell_1589 | 7.87 MiB
Batch 0159 PASSED | cell_1590–cell_1599 | 8.23 MiB
Batch 0160 PASSED | cell_1600–cell_1609 | 8.51 MiB
Batch 0161 PASSED | cell_1610–cell_1619 | 8.61 MiB
Batch 0162 PASSED | cell_1620–cell_1629 | 7.55 MiB
Batch 0163 PASSED | cell_1630–cell_1639 | 8.11 MiB
Batch 0164 PASSED | cell_1640–cell_1649 | 8.33 MiB
Batch 0165 PASSED | cell_1650–c

In [43]:

# ============================================================
# WI-03 — CELL 29
# Audit all 166 registered batches
# ============================================================

import json
import pyarrow.parquet as pq

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

registered = sorted(
    manifest["completed_batches"],
    key=lambda item: item["batch_index"]
)

assert len(registered) == 166
assert [x["batch_index"] for x in registered] == list(range(166))

observed_cells = set()
total_rows = 0
total_size = 0

for item in registered:
    path = FEATURE_DATASET_DIR / item["file_name"]

    assert path.exists(), f"Missing file: {path}"

    table = pq.read_table(path)

    assert table.column_names == EXPECTED_COLUMNS
    assert table.num_rows == item["row_count"]

    keys = table.select(["cell_id", "date"]).to_pandas()
    assert not keys.duplicated(["cell_id", "date"]).any()

    cells = set(keys["cell_id"].unique())
    assert len(cells) == item["cell_count"]
    assert not observed_cells.intersection(cells)

    observed_cells.update(cells)
    total_rows += table.num_rows
    total_size += path.stat().st_size

assert observed_cells == set(all_cells[:1660])
assert total_rows == 1660 * 4383

print("Audited batches:", len(registered))
print("Grid cells:", len(observed_cells))
print("Observations:", f"{total_rows:,}")
print(f"Parquet storage: {total_size / 1024**2:.2f} MiB")
print("Missing files: 0")
print("Duplicate cell/date keys: 0")
print("Schema inconsistencies: 0")
print("Cell coverage audit: PASSED")

Audited batches: 166
Grid cells: 1660
Observations: 7,275,780
Parquet storage: 1353.50 MiB
Missing files: 0
Duplicate cell/date keys: 0
Schema inconsistencies: 0
Cell coverage audit: PASSED


In [44]:

# ============================================================
# WI-03 — CELL 30
# Process batches 0166–0185
# ============================================================

START_BATCH = 166
END_BATCH = 186  # exclusive
MIN_FREE_GB = 3.0

import gc
import json
import shutil
import pyarrow as pa
import pyarrow.parquet as pq

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

completed_indices = {
    item["batch_index"] for item in manifest["completed_batches"]
}

for batch_index in range(START_BATCH, END_BATCH):

    if batch_index in completed_indices:
        print(f"Batch {batch_index:04d} already registered — skipping.")
        continue

    free_gb = shutil.disk_usage(PROJECT_ROOT).free / (1024 ** 3)
    assert free_gb >= MIN_FREE_GB, (
        f"Free disk space below safety threshold: {free_gb:.2f} GiB"
    )

    cells = all_cells[
        batch_index * 10:(batch_index + 1) * 10
    ]

    assert len(cells) > 0, f"No cells for batch {batch_index}"

    batch_path = (
        FEATURE_DATASET_DIR /
        f"batch_{batch_index:04d}.parquet"
    )

    assert not batch_path.exists(), (
        f"Unregistered output already exists: {batch_path}. "
        "Inspect it before proceeding; do not overwrite."
    )

    frame = build_feature_batch(cells)
    expected_rows = len(cells) * 4383

    assert len(frame) == expected_rows
    assert frame.columns.tolist() == EXPECTED_COLUMNS
    assert not frame.duplicated(["cell_id", "date"]).any()

    table = pa.Table.from_pandas(frame, preserve_index=False)

    pq.write_table(
        table,
        batch_path,
        compression="zstd",
        compression_level=3
    )

    check = pq.read_table(batch_path)

    assert check.num_rows == expected_rows
    assert check.column_names == EXPECTED_COLUMNS
    assert set(check.column("cell_id").to_pylist()) == set(cells)

    manifest["completed_batches"].append({
        "batch_index": batch_index,
        "file_name": batch_path.name,
        "cell_count": len(cells),
        "row_count": expected_rows,
        "size_bytes": batch_path.stat().st_size
    })

    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    completed_indices.add(batch_index)

    print(
        f"Batch {batch_index:04d} PASSED | "
        f"{cells[0]}–{cells[-1]} | "
        f"{batch_path.stat().st_size / 1024**2:.2f} MiB"
    )

    del frame, table, check
    gc.collect()

print("\nBlock completed.")
print("Registered batches:", len(completed_indices))


Batch 0166 PASSED | cell_1660–cell_1669 | 8.05 MiB
Batch 0167 PASSED | cell_1670–cell_1679 | 7.86 MiB
Batch 0168 PASSED | cell_1680–cell_1689 | 8.17 MiB
Batch 0169 PASSED | cell_1690–cell_1699 | 8.51 MiB
Batch 0170 PASSED | cell_1700–cell_1709 | 8.75 MiB
Batch 0171 PASSED | cell_1710–cell_1719 | 7.53 MiB
Batch 0172 PASSED | cell_1720–cell_1729 | 8.10 MiB
Batch 0173 PASSED | cell_1730–cell_1739 | 8.36 MiB
Batch 0174 PASSED | cell_1740–cell_1749 | 8.63 MiB
Batch 0175 PASSED | cell_1750–cell_1759 | 8.20 MiB
Batch 0176 PASSED | cell_1760–cell_1769 | 7.94 MiB
Batch 0177 PASSED | cell_1770–cell_1779 | 8.24 MiB
Batch 0178 PASSED | cell_1780–cell_1789 | 8.58 MiB
Batch 0179 PASSED | cell_1790–cell_1799 | 8.66 MiB
Batch 0180 PASSED | cell_1800–cell_1809 | 7.60 MiB
Batch 0181 PASSED | cell_1810–cell_1819 | 8.15 MiB
Batch 0182 PASSED | cell_1820–cell_1829 | 8.58 MiB
Batch 0183 PASSED | cell_1830–cell_1839 | 8.62 MiB
Batch 0184 PASSED | cell_1840–cell_1849 | 8.36 MiB
Batch 0185 PASSED | cell_1850–c

In [45]:

# ============================================================
# WI-03 — CELL 31
# Audit all 186 registered batches
# ============================================================

import json
import pyarrow.parquet as pq

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

registered = sorted(
    manifest["completed_batches"],
    key=lambda item: item["batch_index"]
)

assert len(registered) == 186
assert [x["batch_index"] for x in registered] == list(range(186))

observed_cells = set()
total_rows = 0
total_size = 0

for item in registered:
    path = FEATURE_DATASET_DIR / item["file_name"]

    assert path.exists(), f"Missing file: {path}"

    table = pq.read_table(path)

    assert table.column_names == EXPECTED_COLUMNS
    assert table.num_rows == item["row_count"]

    keys = table.select(["cell_id", "date"]).to_pandas()
    assert not keys.duplicated(["cell_id", "date"]).any()

    cells = set(keys["cell_id"].unique())
    assert len(cells) == item["cell_count"]
    assert not observed_cells.intersection(cells)

    observed_cells.update(cells)
    total_rows += table.num_rows
    total_size += path.stat().st_size

assert observed_cells == set(all_cells[:1860])
assert total_rows == 1860 * 4383

print("Audited batches:", len(registered))
print("Grid cells:", len(observed_cells))
print("Observations:", f"{total_rows:,}")
print(f"Parquet storage: {total_size / 1024**2:.2f} MiB")
print("Missing files: 0")
print("Duplicate cell/date keys: 0")
print("Schema inconsistencies: 0")
print("Cell coverage audit: PASSED")


Audited batches: 186
Grid cells: 1860
Observations: 8,152,380
Parquet storage: 1518.38 MiB
Missing files: 0
Duplicate cell/date keys: 0
Schema inconsistencies: 0
Cell coverage audit: PASSED


In [46]:

# ============================================================
# WI-03 — CELL 32
# Process batches 0186–0205
# ============================================================

START_BATCH = 186
END_BATCH = 206  # exclusive
MIN_FREE_GB = 3.0

import gc
import json
import shutil
import pyarrow as pa
import pyarrow.parquet as pq

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

completed_indices = {
    item["batch_index"] for item in manifest["completed_batches"]
}

for batch_index in range(START_BATCH, END_BATCH):

    if batch_index in completed_indices:
        print(f"Batch {batch_index:04d} already registered — skipping.")
        continue

    free_gb = shutil.disk_usage(PROJECT_ROOT).free / (1024 ** 3)
    assert free_gb >= MIN_FREE_GB, (
        f"Free disk space below safety threshold: {free_gb:.2f} GiB"
    )

    cells = all_cells[
        batch_index * 10:(batch_index + 1) * 10
    ]

    assert len(cells) > 0, f"No cells for batch {batch_index}"

    batch_path = (
        FEATURE_DATASET_DIR /
        f"batch_{batch_index:04d}.parquet"
    )

    assert not batch_path.exists(), (
        f"Unregistered output already exists: {batch_path}. "
        "Inspect it before proceeding; do not overwrite."
    )

    frame = build_feature_batch(cells)
    expected_rows = len(cells) * 4383

    assert len(frame) == expected_rows
    assert frame.columns.tolist() == EXPECTED_COLUMNS
    assert not frame.duplicated(["cell_id", "date"]).any()

    table = pa.Table.from_pandas(frame, preserve_index=False)

    pq.write_table(
        table,
        batch_path,
        compression="zstd",
        compression_level=3
    )

    check = pq.read_table(batch_path)

    assert check.num_rows == expected_rows
    assert check.column_names == EXPECTED_COLUMNS
    assert set(check.column("cell_id").to_pylist()) == set(cells)

    manifest["completed_batches"].append({
        "batch_index": batch_index,
        "file_name": batch_path.name,
        "cell_count": len(cells),
        "row_count": expected_rows,
        "size_bytes": batch_path.stat().st_size
    })

    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    completed_indices.add(batch_index)

    print(
        f"Batch {batch_index:04d} PASSED | "
        f"{cells[0]}–{cells[-1]} | "
        f"{batch_path.stat().st_size / 1024**2:.2f} MiB"
    )

    del frame, table, check
    gc.collect()

print("\nBlock completed.")
print("Registered batches:", len(completed_indices))


Batch 0186 PASSED | cell_1860–cell_1869 | 8.45 MiB
Batch 0187 PASSED | cell_1870–cell_1879 | 8.79 MiB
Batch 0188 PASSED | cell_1880–cell_1889 | 8.99 MiB
Batch 0189 PASSED | cell_1890–cell_1899 | 7.71 MiB
Batch 0190 PASSED | cell_1900–cell_1909 | 8.27 MiB
Batch 0191 PASSED | cell_1910–cell_1919 | 8.97 MiB
Batch 0192 PASSED | cell_1920–cell_1929 | 8.82 MiB
Batch 0193 PASSED | cell_1930–cell_1939 | 8.57 MiB
Batch 0194 PASSED | cell_1940–cell_1949 | 8.02 MiB
Batch 0195 PASSED | cell_1950–cell_1959 | 8.84 MiB
Batch 0196 PASSED | cell_1960–cell_1969 | 8.68 MiB
Batch 0197 PASSED | cell_1970–cell_1979 | 9.06 MiB
Batch 0198 PASSED | cell_1980–cell_1989 | 7.72 MiB
Batch 0199 PASSED | cell_1990–cell_1999 | 8.42 MiB
Batch 0200 PASSED | cell_2000–cell_2009 | 8.88 MiB
Batch 0201 PASSED | cell_2010–cell_2019 | 8.62 MiB
Batch 0202 PASSED | cell_2020–cell_2029 | 8.68 MiB
Batch 0203 PASSED | cell_2030–cell_2039 | 8.14 MiB
Batch 0204 PASSED | cell_2040–cell_2049 | 9.00 MiB
Batch 0205 PASSED | cell_2050–c

In [47]:

# ============================================================
# WI-03 — CELL 33
# Audit all 206 registered batches
# ============================================================

import json
import pyarrow.parquet as pq

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

registered = sorted(
    manifest["completed_batches"],
    key=lambda item: item["batch_index"]
)

assert len(registered) == 206
assert [x["batch_index"] for x in registered] == list(range(206))

observed_cells = set()
total_rows = 0
total_size = 0

for item in registered:
    path = FEATURE_DATASET_DIR / item["file_name"]

    assert path.exists(), f"Missing file: {path}"

    table = pq.read_table(path)

    assert table.column_names == EXPECTED_COLUMNS
    assert table.num_rows == item["row_count"]

    keys = table.select(["cell_id", "date"]).to_pandas()
    assert not keys.duplicated(["cell_id", "date"]).any()

    cells = set(keys["cell_id"].unique())
    assert len(cells) == item["cell_count"]
    assert not observed_cells.intersection(cells)

    observed_cells.update(cells)
    total_rows += table.num_rows
    total_size += path.stat().st_size

assert observed_cells == set(all_cells[:2060])
assert total_rows == 2060 * 4383

print("Audited batches:", len(registered))
print("Grid cells:", len(observed_cells))
print("Observations:", f"{total_rows:,}")
print(f"Parquet storage: {total_size / 1024**2:.2f} MiB")
print("Missing files: 0")
print("Duplicate cell/date keys: 0")
print("Schema inconsistencies: 0")
print("Cell coverage audit: PASSED")


Audited batches: 206
Grid cells: 2060
Observations: 9,028,980
Parquet storage: 1689.61 MiB
Missing files: 0
Duplicate cell/date keys: 0
Schema inconsistencies: 0
Cell coverage audit: PASSED


In [48]:

# ============================================================
# WI-03 — CELL 34
# Process batches 0206–0225
# ============================================================

START_BATCH = 206
END_BATCH = 226  # exclusive
MIN_FREE_GB = 3.0

import gc
import json
import shutil
import pyarrow as pa
import pyarrow.parquet as pq

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

completed_indices = {
    item["batch_index"] for item in manifest["completed_batches"]
}

for batch_index in range(START_BATCH, END_BATCH):

    if batch_index in completed_indices:
        print(f"Batch {batch_index:04d} already registered — skipping.")
        continue

    free_gb = shutil.disk_usage(PROJECT_ROOT).free / (1024 ** 3)
    assert free_gb >= MIN_FREE_GB, (
        f"Free disk space below safety threshold: {free_gb:.2f} GiB"
    )

    cells = all_cells[
        batch_index * 10:(batch_index + 1) * 10
    ]

    assert len(cells) > 0, f"No cells for batch {batch_index}"

    batch_path = (
        FEATURE_DATASET_DIR /
        f"batch_{batch_index:04d}.parquet"
    )

    assert not batch_path.exists(), (
        f"Unregistered output already exists: {batch_path}. "
        "Inspect it before proceeding; do not overwrite."
    )

    frame = build_feature_batch(cells)
    expected_rows = len(cells) * 4383

    assert len(frame) == expected_rows
    assert frame.columns.tolist() == EXPECTED_COLUMNS
    assert not frame.duplicated(["cell_id", "date"]).any()

    table = pa.Table.from_pandas(frame, preserve_index=False)

    pq.write_table(
        table,
        batch_path,
        compression="zstd",
        compression_level=3
    )

    check = pq.read_table(batch_path)

    assert check.num_rows == expected_rows
    assert check.column_names == EXPECTED_COLUMNS
    assert set(check.column("cell_id").to_pylist()) == set(cells)

    manifest["completed_batches"].append({
        "batch_index": batch_index,
        "file_name": batch_path.name,
        "cell_count": len(cells),
        "row_count": expected_rows,
        "size_bytes": batch_path.stat().st_size
    })

    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    completed_indices.add(batch_index)

    print(
        f"Batch {batch_index:04d} PASSED | "
        f"{cells[0]}–{cells[-1]} | "
        f"{batch_path.stat().st_size / 1024**2:.2f} MiB"
    )

    del frame, table, check
    gc.collect()

print("\nBlock completed.")
print("Registered batches:", len(completed_indices))


Batch 0206 PASSED | cell_2060–cell_2069 | 8.89 MiB
Batch 0207 PASSED | cell_2070–cell_2079 | 7.94 MiB
Batch 0208 PASSED | cell_2080–cell_2089 | 8.77 MiB
Batch 0209 PASSED | cell_2090–cell_2099 | 8.71 MiB
Batch 0210 PASSED | cell_2100–cell_2109 | 8.59 MiB
Batch 0211 PASSED | cell_2110–cell_2119 | 8.72 MiB
Batch 0212 PASSED | cell_2120–cell_2129 | 8.35 MiB
Batch 0213 PASSED | cell_2130–cell_2139 | 8.89 MiB
Batch 0214 PASSED | cell_2140–cell_2149 | 8.44 MiB
Batch 0215 PASSED | cell_2150–cell_2159 | 8.87 MiB
Batch 0216 PASSED | cell_2160–cell_2169 | 8.10 MiB
Batch 0217 PASSED | cell_2170–cell_2179 | 8.97 MiB
Batch 0218 PASSED | cell_2180–cell_2189 | 8.58 MiB
Batch 0219 PASSED | cell_2190–cell_2199 | 8.60 MiB
Batch 0220 PASSED | cell_2200–cell_2209 | 8.86 MiB
Batch 0221 PASSED | cell_2210–cell_2219 | 8.75 MiB
Batch 0222 PASSED | cell_2220–cell_2229 | 8.60 MiB
Batch 0223 PASSED | cell_2230–cell_2239 | 8.41 MiB
Batch 0224 PASSED | cell_2240–cell_2249 | 8.91 MiB
Batch 0225 PASSED | cell_2250–c

In [49]:

# ============================================================
# WI-03 — CELL 35
# Audit all 226 registered batches
# ============================================================

import json
import pyarrow.parquet as pq

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

registered = sorted(
    manifest["completed_batches"],
    key=lambda item: item["batch_index"]
)

assert len(registered) == 226
assert [x["batch_index"] for x in registered] == list(range(226))

observed_cells = set()
total_rows = 0
total_size = 0

for item in registered:
    path = FEATURE_DATASET_DIR / item["file_name"]

    assert path.exists(), f"Missing file: {path}"

    table = pq.read_table(path)

    assert table.column_names == EXPECTED_COLUMNS
    assert table.num_rows == item["row_count"]

    keys = table.select(["cell_id", "date"]).to_pandas()
    assert not keys.duplicated(["cell_id", "date"]).any()

    cells = set(keys["cell_id"].unique())
    assert len(cells) == item["cell_count"]
    assert not observed_cells.intersection(cells)

    observed_cells.update(cells)
    total_rows += table.num_rows
    total_size += path.stat().st_size

assert observed_cells == set(all_cells[:2260])
assert total_rows == 2260 * 4383

print("Audited batches:", len(registered))
print("Grid cells:", len(observed_cells))
print("Observations:", f"{total_rows:,}")
print(f"Parquet storage: {total_size / 1024**2:.2f} MiB")
print("Missing files: 0")
print("Duplicate cell/date keys: 0")
print("Schema inconsistencies: 0")
print("Cell coverage audit: PASSED")


Audited batches: 226
Grid cells: 2260
Observations: 9,905,580
Parquet storage: 1862.00 MiB
Missing files: 0
Duplicate cell/date keys: 0
Schema inconsistencies: 0
Cell coverage audit: PASSED


In [50]:

# ============================================================
# WI-03 — CELL 36
# Process final batches 0226–0251
# ============================================================

START_BATCH = 226
END_BATCH = 252  # exclusive
MIN_FREE_GB = 3.0

import gc
import json
import shutil
import pyarrow as pa
import pyarrow.parquet as pq

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

completed_indices = {
    item["batch_index"] for item in manifest["completed_batches"]
}

for batch_index in range(START_BATCH, END_BATCH):

    if batch_index in completed_indices:
        print(f"Batch {batch_index:04d} already registered — skipping.")
        continue

    free_gb = shutil.disk_usage(PROJECT_ROOT).free / (1024 ** 3)
    assert free_gb >= MIN_FREE_GB, (
        f"Free disk space below safety threshold: {free_gb:.2f} GiB"
    )

    cells = all_cells[
        batch_index * 10:(batch_index + 1) * 10
    ]

    assert len(cells) > 0, f"No cells for batch {batch_index}"

    batch_path = (
        FEATURE_DATASET_DIR /
        f"batch_{batch_index:04d}.parquet"
    )

    assert not batch_path.exists(), (
        f"Unregistered output already exists: {batch_path}. "
        "Inspect it before proceeding; do not overwrite."
    )

    frame = build_feature_batch(cells)
    expected_rows = len(cells) * 4383

    assert len(frame) == expected_rows
    assert frame.columns.tolist() == EXPECTED_COLUMNS
    assert not frame.duplicated(["cell_id", "date"]).any()

    table = pa.Table.from_pandas(frame, preserve_index=False)

    pq.write_table(
        table,
        batch_path,
        compression="zstd",
        compression_level=3
    )

    check = pq.read_table(batch_path)

    assert check.num_rows == expected_rows
    assert check.column_names == EXPECTED_COLUMNS
    assert set(check.column("cell_id").to_pylist()) == set(cells)

    manifest["completed_batches"].append({
        "batch_index": batch_index,
        "file_name": batch_path.name,
        "cell_count": len(cells),
        "row_count": expected_rows,
        "size_bytes": batch_path.stat().st_size
    })

    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    completed_indices.add(batch_index)

    print(
        f"Batch {batch_index:04d} PASSED | "
        f"{cells[0]}–{cells[-1]} | "
        f"{batch_path.stat().st_size / 1024**2:.2f} MiB"
    )

    del frame, table, check
    gc.collect()

print("\nFinal processing block completed.")
print("Registered batches:", len(completed_indices))


Batch 0226 PASSED | cell_2260–cell_2269 | 9.00 MiB
Batch 0227 PASSED | cell_2270–cell_2279 | 8.42 MiB
Batch 0228 PASSED | cell_2280–cell_2289 | 8.65 MiB
Batch 0229 PASSED | cell_2290–cell_2299 | 9.04 MiB
Batch 0230 PASSED | cell_2300–cell_2309 | 9.07 MiB
Batch 0231 PASSED | cell_2310–cell_2319 | 8.36 MiB
Batch 0232 PASSED | cell_2320–cell_2329 | 8.50 MiB
Batch 0233 PASSED | cell_2330–cell_2339 | 8.87 MiB
Batch 0234 PASSED | cell_2340–cell_2349 | 8.78 MiB
Batch 0235 PASSED | cell_2350–cell_2359 | 8.80 MiB
Batch 0236 PASSED | cell_2360–cell_2369 | 8.38 MiB
Batch 0237 PASSED | cell_2370–cell_2379 | 8.68 MiB
Batch 0238 PASSED | cell_2380–cell_2389 | 9.14 MiB
Batch 0239 PASSED | cell_2390–cell_2399 | 9.22 MiB
Batch 0240 PASSED | cell_2400–cell_2409 | 8.18 MiB
Batch 0241 PASSED | cell_2410–cell_2419 | 8.51 MiB
Batch 0242 PASSED | cell_2420–cell_2429 | 8.84 MiB
Batch 0243 PASSED | cell_2430–cell_2439 | 9.07 MiB
Batch 0244 PASSED | cell_2440–cell_2449 | 8.60 MiB
Batch 0245 PASSED | cell_2450–c

In [52]:

# ============================================================
# WI-03 — CELL 37A
# Corrected full-archive audit
# ============================================================

import json
import pyarrow.parquet as pq

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

registered = sorted(
    manifest["completed_batches"],
    key=lambda item: item["batch_index"]
)

assert len(registered) == 252
assert [x["batch_index"] for x in registered] == list(range(252))

observed_cells = set()
total_rows = 0
total_size = 0
total_files = 0
missing_files = []
row_mismatches = []
schema_errors = []
duplicate_errors = []
coverage_errors = []
manifest_size_fields = 0

for item in registered:
    batch_index = item["batch_index"]
    path = FEATURE_DATASET_DIR / item["file_name"]

    if not path.exists():
        missing_files.append(item["file_name"])
        continue

    actual_size = path.stat().st_size
    total_size += actual_size
    total_files += 1

    # Validate size only when the manifest contains it
    if "size_bytes" in item:
        manifest_size_fields += 1
        if actual_size != item["size_bytes"]:
            row_mismatches.append(
                f"Size mismatch: {item['file_name']}"
            )

    table = pq.read_table(path)

    expected_cells = all_cells[
        batch_index * 10:(batch_index + 1) * 10
    ]
    expected_rows = len(expected_cells) * 4383

    if table.column_names != EXPECTED_COLUMNS:
        schema_errors.append(item["file_name"])

    if (
        table.num_rows != expected_rows
        or table.num_rows != item["row_count"]
    ):
        row_mismatches.append(item["file_name"])

    keys = table.select(["cell_id", "date"]).to_pandas()

    if keys.duplicated(["cell_id", "date"]).any():
        duplicate_errors.append(item["file_name"])

    batch_cells = set(keys["cell_id"].unique())

    if batch_cells != set(expected_cells):
        coverage_errors.append(item["file_name"])

    if observed_cells.intersection(batch_cells):
        coverage_errors.append(
            f"Overlapping cells: {item['file_name']}"
        )

    per_cell_counts = keys.groupby("cell_id").size()

    if not (per_cell_counts == 4383).all():
        coverage_errors.append(
            f"Incomplete daily coverage: {item['file_name']}"
        )

    observed_cells.update(batch_cells)
    total_rows += table.num_rows

# Global assertions
assert not missing_files, missing_files
assert not row_mismatches, row_mismatches
assert not schema_errors, schema_errors
assert not duplicate_errors, duplicate_errors
assert not coverage_errors, coverage_errors
assert total_files == 252
assert len(observed_cells) == 2520
assert observed_cells == set(all_cells)
assert total_rows == EXPECTED_OBSERVATIONS

print("FULL ARCHIVE AUDIT: PASSED")
print("Registered batches:", total_files)
print("Grid cells:", len(observed_cells))
print("Observations:", f"{total_rows:,}")
print("Feature columns:", len(FEATURE_COLUMNS))
print("Target columns:", len(TARGET_COLUMNS))
print("Total columns:", len(EXPECTED_COLUMNS))
print(f"Parquet storage: {total_size / 1024**2:.2f} MiB")
print("Manifest entries with size metadata:", manifest_size_fields)
print("Missing files: 0")
print("Row count mismatches: 0")
print("Schema inconsistencies: 0")
print("Duplicate cell/date keys: 0")
print("Cell coverage gaps/overlaps: 0")
print("Per-cell daily coverage: PASSED")


FULL ARCHIVE AUDIT: PASSED
Registered batches: 252
Grid cells: 2520
Observations: 11,045,160
Feature columns: 53
Target columns: 14
Total columns: 69
Parquet storage: 2088.66 MiB
Manifest entries with size metadata: 166
Missing files: 0
Row count mismatches: 0
Schema inconsistencies: 0
Duplicate cell/date keys: 0
Cell coverage gaps/overlaps: 0
Per-cell daily coverage: PASSED


In [53]:

# ============================================================
# WI-03 — CELL 38
# Normalize batch manifest metadata
# ============================================================

import json
import shutil

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

entries = sorted(
    manifest["completed_batches"],
    key=lambda item: item["batch_index"]
)

assert len(entries) == 252
assert [x["batch_index"] for x in entries] == list(range(252))

# Preserve the original manifest before modification
backup_path = manifest_path.with_name(
    manifest_path.stem + "_pre_size_backfill_backup.json"
)

assert not backup_path.exists(), (
    f"Backup already exists; inspect before proceeding: {backup_path}"
)

shutil.copy2(manifest_path, backup_path)

updated_count = 0
total_size = 0

for item in entries:
    path = FEATURE_DATASET_DIR / item["file_name"]

    assert path.exists(), f"Missing file: {path}"

    actual_size = path.stat().st_size

    if item.get("size_bytes") != actual_size:
        updated_count += 1

    item["size_bytes"] = actual_size
    total_size += actual_size

# Save normalized manifest
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

# Read back and verify
with open(manifest_path, "r", encoding="utf-8") as f:
    verified_manifest = json.load(f)

verified_entries = verified_manifest["completed_batches"]

assert len(verified_entries) == 252
assert all("size_bytes" in x for x in verified_entries)

for item in verified_entries:
    path = FEATURE_DATASET_DIR / item["file_name"]
    assert path.stat().st_size == item["size_bytes"]

print("MANIFEST NORMALIZATION: PASSED")
print("Manifest entries:", len(verified_entries))
print("Entries updated:", updated_count)
print("Entries with size metadata:", sum(
    "size_bytes" in x for x in verified_entries
))
print(f"Total Parquet storage: {total_size / 1024**2:.2f} MiB")
print("Original manifest backup:", backup_path.name)
print("Parquet files modified: 0")


MANIFEST NORMALIZATION: PASSED
Manifest entries: 252
Entries updated: 86
Entries with size metadata: 252
Total Parquet storage: 2088.66 MiB
Original manifest backup: batch_manifest_pre_size_backfill_backup.json
Parquet files modified: 0


In [54]:

# ============================================================
# WI-03 — CELL 39
# Full dataset feature/target quality audit
# ============================================================

import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

feature_missing = {c: 0 for c in FEATURE_COLUMNS}
target_missing = {c: 0 for c in TARGET_COLUMNS}
nonfinite_counts = {c: 0 for c in FEATURE_COLUMNS + TARGET_COLUMNS}

total_rows = 0
feature_complete_rows = 0
target_complete_rows = 0
fully_complete_rows = 0

for batch_index in range(252):
    path = FEATURE_DATASET_DIR / f"batch_{batch_index:04d}.parquet"

    frame = pq.read_table(path).to_pandas()
    total_rows += len(frame)

    feature_na = frame[FEATURE_COLUMNS].isna()
    target_na = frame[TARGET_COLUMNS].isna()

    for col in FEATURE_COLUMNS:
        feature_missing[col] += int(feature_na[col].sum())

    for col in TARGET_COLUMNS:
        target_missing[col] += int(target_na[col].sum())

    numeric_columns = FEATURE_COLUMNS + TARGET_COLUMNS

    for col in numeric_columns:
        values = frame[col].dropna()

        if pd.api.types.is_numeric_dtype(values):
            nonfinite_counts[col] += int(
                (~np.isfinite(values.to_numpy(dtype=float))).sum()
            )

    feature_complete = ~feature_na.any(axis=1)
    target_complete = ~target_na.any(axis=1)

    feature_complete_rows += int(feature_complete.sum())
    target_complete_rows += int(target_complete.sum())
    fully_complete_rows += int(
        (feature_complete & target_complete).sum()
    )

    del frame, feature_na, target_na

assert total_rows == EXPECTED_OBSERVATIONS

feature_missing_df = pd.DataFrame([
    {
        "feature": col,
        "missing_count": count,
        "missing_pct": count / total_rows * 100
    }
    for col, count in feature_missing.items()
])

target_missing_df = pd.DataFrame([
    {
        "target": col,
        "missing_count": count,
        "missing_pct": count / total_rows * 100
    }
    for col, count in target_missing.items()
])

nonfinite_df = pd.DataFrame([
    {
        "column": col,
        "nonfinite_count": count
    }
    for col, count in nonfinite_counts.items()
])

print("DATASET QUALITY AUDIT")
print("Total observations:", f"{total_rows:,}")
print(
    "Feature-complete rows:",
    f"{feature_complete_rows:,}",
    f"({feature_complete_rows / total_rows * 100:.3f}%)"
)
print(
    "Target-complete rows:",
    f"{target_complete_rows:,}",
    f"({target_complete_rows / total_rows * 100:.3f}%)"
)
print(
    "Fully complete rows:",
    f"{fully_complete_rows:,}",
    f"({fully_complete_rows / total_rows * 100:.3f}%)"
)

print("\nFEATURE MISSINGNESS")
display(
    feature_missing_df.sort_values(
        "missing_count", ascending=False
    ).head(20)
)

print("\nTARGET MISSINGNESS")
display(
    target_missing_df.sort_values(
        "missing_count", ascending=False
    )
)

print("\nNON-FINITE VALUES")
display(
    nonfinite_df.sort_values(
        "nonfinite_count", ascending=False
    ).head(20)
)

# Preserve audit results
quality_report = {
    "total_rows": total_rows,
    "feature_complete_rows": feature_complete_rows,
    "target_complete_rows": target_complete_rows,
    "fully_complete_rows": fully_complete_rows,
    "feature_missingness": feature_missing,
    "target_missingness": target_missing,
    "nonfinite_counts": nonfinite_counts
}

quality_path = REPORT_DIR / "WI_03_feature_quality_audit.json"

with open(quality_path, "w", encoding="utf-8") as f:
    json.dump(quality_report, f, indent=2)

print("\nQuality report saved:", quality_path)


DATASET QUALITY AUDIT
Total observations: 11,045,160
Feature-complete rows: 10,969,560 (99.316%)
Target-complete rows: 11,027,520 (99.840%)
Fully complete rows: 10,951,920 (99.156%)

FEATURE MISSINGNESS


,feature,missing_count,missing_pct
10,PRECTOTCORR_lag_30,75600,0.684463
16,T2M_MAX_lag_30,75600,0.684463
22,T2M_MIN_lag_30,75600,0.684463
29,PRECTOTCORR_rolling_mean_30,73080,0.661647
30,PRECTOTCORR_rolling_std_30,73080,0.661647
37,T2M_MAX_rolling_mean_30,73080,0.661647
38,T2M_MAX_rolling_std_30,73080,0.661647
46,T2M_MIN_rolling_std_30,73080,0.661647
45,T2M_MIN_rolling_mean_30,73080,0.661647
21,T2M_MIN_lag_14,35280,0.319416



TARGET MISSINGNESS


,target,missing_count,missing_pct
13,T2M_MAX_target_t_plus_7,17640,0.159708
6,PRECTOTCORR_target_t_plus_7,17640,0.159708
5,PRECTOTCORR_target_t_plus_6,15120,0.136893
12,T2M_MAX_target_t_plus_6,15120,0.136893
11,T2M_MAX_target_t_plus_5,12600,0.114077
4,PRECTOTCORR_target_t_plus_5,12600,0.114077
10,T2M_MAX_target_t_plus_4,10080,0.091262
3,PRECTOTCORR_target_t_plus_4,10080,0.091262
2,PRECTOTCORR_target_t_plus_3,7560,0.068446
9,T2M_MAX_target_t_plus_3,7560,0.068446



NON-FINITE VALUES


,column,nonfinite_count
0,latitude,0
1,longitude,0
2,PRECTOTCORR_current,0
3,T2M_MAX_current,0
4,T2M_MIN_current,0
5,PRECTOTCORR_lag_1,0
6,PRECTOTCORR_lag_2,0
7,PRECTOTCORR_lag_3,0
8,PRECTOTCORR_lag_7,0
9,PRECTOTCORR_lag_14,0



Quality report saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_03_feature_quality_audit.json


In [55]:

# ============================================================
# WI-03 — CELL 40
# Verify structural missingness by date
# ============================================================

import pandas as pd
import pyarrow.parquet as pq
from collections import defaultdict

feature_missing_dates = defaultdict(list)
target_missing_dates = defaultdict(list)

for batch_index in range(252):
    path = FEATURE_DATASET_DIR / f"batch_{batch_index:04d}.parquet"
    frame = pq.read_table(path).to_pandas()

    for col in FEATURE_COLUMNS:
        missing = frame.loc[frame[col].isna(), ["cell_id", "date"]]
        if not missing.empty:
            feature_missing_dates[col].append(missing)

    for col in TARGET_COLUMNS:
        missing = frame.loc[frame[col].isna(), ["cell_id", "date"]]
        if not missing.empty:
            target_missing_dates[col].append(missing)

    del frame

feature_date_rows = []

for col, parts in feature_missing_dates.items():
    missing = pd.concat(parts, ignore_index=True)

    per_cell = missing.groupby("cell_id")["date"].agg(
        missing_days="count",
        first_missing="min",
        last_missing="max"
    )

    feature_date_rows.append({
        "feature": col,
        "missing_rows": len(missing),
        "cells_affected": missing["cell_id"].nunique(),
        "min_missing_days_per_cell": per_cell["missing_days"].min(),
        "max_missing_days_per_cell": per_cell["missing_days"].max(),
        "all_at_series_start": bool(
            (per_cell["first_missing"] == pd.Timestamp("2013-01-01")).all()
        ),
        "max_missing_date": missing["date"].max()
    })

target_date_rows = []

for col, parts in target_missing_dates.items():
    missing = pd.concat(parts, ignore_index=True)

    per_cell = missing.groupby("cell_id")["date"].agg(
        missing_days="count",
        first_missing="min",
        last_missing="max"
    )

    target_date_rows.append({
        "target": col,
        "missing_rows": len(missing),
        "cells_affected": missing["cell_id"].nunique(),
        "min_missing_days_per_cell": per_cell["missing_days"].min(),
        "max_missing_days_per_cell": per_cell["missing_days"].max(),
        "all_at_series_end": bool(
            (per_cell["last_missing"] == pd.Timestamp("2024-12-31")).all()
        ),
        "min_missing_date": missing["date"].min()
    })

feature_boundary_df = pd.DataFrame(feature_date_rows)
target_boundary_df = pd.DataFrame(target_date_rows)

print("FEATURE MISSINGNESS BY DATE")
display(feature_boundary_df.sort_values("feature"))

print("\nTARGET MISSINGNESS BY DATE")
display(target_boundary_df.sort_values("target"))

print("\nFeature missing rows:", sum(
    len(pd.concat(parts, ignore_index=True))
    for parts in feature_missing_dates.values()
))

print("Target missing rows:", sum(
    len(pd.concat(parts, ignore_index=True))
    for parts in target_missing_dates.values()
))


FEATURE MISSINGNESS BY DATE


,feature,missing_rows,cells_affected,min_missing_days_per_cell,max_missing_days_per_cell,all_at_series_start,max_missing_date
0,PRECTOTCORR_lag_1,2520,2520,1,1,True,2013-01-01
4,PRECTOTCORR_lag_14,35280,2520,14,14,True,2013-01-14
1,PRECTOTCORR_lag_2,5040,2520,2,2,True,2013-01-02
2,PRECTOTCORR_lag_3,7560,2520,3,3,True,2013-01-03
5,PRECTOTCORR_lag_30,75600,2520,30,30,True,2013-01-30
3,PRECTOTCORR_lag_7,17640,2520,7,7,True,2013-01-07
22,PRECTOTCORR_rolling_mean_14,32760,2520,13,13,True,2013-01-13
18,PRECTOTCORR_rolling_mean_3,5040,2520,2,2,True,2013-01-02
24,PRECTOTCORR_rolling_mean_30,73080,2520,29,29,True,2013-01-29
20,PRECTOTCORR_rolling_mean_7,15120,2520,6,6,True,2013-01-06



TARGET MISSINGNESS BY DATE


,target,missing_rows,cells_affected,min_missing_days_per_cell,max_missing_days_per_cell,all_at_series_end,min_missing_date
0,PRECTOTCORR_target_t_plus_1,2520,2520,1,1,True,2024-12-31
1,PRECTOTCORR_target_t_plus_2,5040,2520,2,2,True,2024-12-30
2,PRECTOTCORR_target_t_plus_3,7560,2520,3,3,True,2024-12-29
3,PRECTOTCORR_target_t_plus_4,10080,2520,4,4,True,2024-12-28
4,PRECTOTCORR_target_t_plus_5,12600,2520,5,5,True,2024-12-27
5,PRECTOTCORR_target_t_plus_6,15120,2520,6,6,True,2024-12-26
6,PRECTOTCORR_target_t_plus_7,17640,2520,7,7,True,2024-12-25
7,T2M_MAX_target_t_plus_1,2520,2520,1,1,True,2024-12-31
8,T2M_MAX_target_t_plus_2,5040,2520,2,2,True,2024-12-30
9,T2M_MAX_target_t_plus_3,7560,2520,3,3,True,2024-12-29



Feature missing rows: 1186920
Target missing rows: 141120


In [56]:

# ============================================================
# WI-03 — CELL 41
# Verify target-date split integrity
# ============================================================

import pandas as pd
import pyarrow.parquet as pq

DEV_START = pd.Timestamp("2013-01-01")
DEV_END = pd.Timestamp("2022-12-31")

HOLDOUT_START = pd.Timestamp("2023-01-01")
HOLDOUT_END = pd.Timestamp("2024-12-31")

split_results = []

for horizon in FORECAST_HORIZONS:

    target_col = f"PRECTOTCORR_target_t_plus_{horizon}"

    dev_rows = 0
    holdout_rows = 0
    outside_rows = 0
    missing_target_rows = 0
    boundary_dev_rows = 0

    for batch_index in range(252):
        path = FEATURE_DATASET_DIR / f"batch_{batch_index:04d}.parquet"

        frame = pq.read_table(
            path,
            columns=["date", target_col]
        ).to_pandas()

        origin_date = pd.to_datetime(frame["date"])
        target_date = origin_date + pd.to_timedelta(horizon, unit="D")

        valid_target = frame[target_col].notna()

        dev_mask = (
            valid_target
            & (target_date >= DEV_START)
            & (target_date <= DEV_END)
        )

        holdout_mask = (
            valid_target
            & (target_date >= HOLDOUT_START)
            & (target_date <= HOLDOUT_END)
        )

        outside_mask = valid_target & ~(dev_mask | holdout_mask)

        boundary_mask = (
            dev_mask
            & (origin_date >= HOLDOUT_START)
        )

        dev_rows += int(dev_mask.sum())
        holdout_rows += int(holdout_mask.sum())
        outside_rows += int(outside_mask.sum())
        missing_target_rows += int((~valid_target).sum())
        boundary_dev_rows += int(boundary_mask.sum())

        del frame

    assert outside_rows == 0, (
        f"Unexpected target dates outside configured periods for T+{horizon}"
    )

    split_results.append({
        "horizon": f"T+{horizon}",
        "development_rows": dev_rows,
        "holdout_rows": holdout_rows,
        "missing_target_rows": missing_target_rows,
        "outside_period_rows": outside_rows,
        "invalid_boundary_dev_rows": boundary_dev_rows
    })

split_df = pd.DataFrame(split_results)

display(split_df)

print("\nTARGET-DATE SPLIT AUDIT")
print("Horizons checked:", len(split_results))
print("Unexpected target dates:", int(
    split_df["outside_period_rows"].sum()
))
print("Invalid development boundary rows:", int(
    split_df["invalid_boundary_dev_rows"].sum()
))

assert (split_df["outside_period_rows"] == 0).all()
assert (split_df["invalid_boundary_dev_rows"] == 0).all()

print("TARGET-DATE SPLIT AUDIT: PASSED")


,horizon,development_rows,holdout_rows,missing_target_rows,outside_period_rows,invalid_boundary_dev_rows
0,T+1,9200520,1842120,2520,0,0
1,T+2,9198000,1842120,5040,0,0
2,T+3,9195480,1842120,7560,0,0
3,T+4,9192960,1842120,10080,0,0
4,T+5,9190440,1842120,12600,0,0
5,T+6,9187920,1842120,15120,0,0
6,T+7,9185400,1842120,17640,0,0



TARGET-DATE SPLIT AUDIT
Horizons checked: 7
Unexpected target dates: 0
Invalid development boundary rows: 0
TARGET-DATE SPLIT AUDIT: PASSED


In [57]:

# ============================================================
# WI-03 — CELL 42
# Save chronological split policy
# ============================================================

import json

split_policy = {
    "dataset_version": "weather-features-v1",
    "split_method": "chronological_by_target_date",
    "forecast_origin_convention": (
        "End of day t, after day t weather observations are available"
    ),
    "development": {
        "start": "2013-01-01",
        "end": "2022-12-31",
        "purpose": "Model development and chronological validation"
    },
    "final_holdout": {
        "start": "2023-01-01",
        "end": "2024-12-31",
        "purpose": "Final untouched evaluation"
    },
    "horizons": FORECAST_HORIZONS,
    "targets": TARGET_VARIABLES,
    "target_date_rule": "target_date = forecast_origin_date + horizon_days",
    "row_assignment": (
        "Assign each forecast row independently for each horizon "
        "using its target date"
    ),
    "missing_target_policy": (
        "Exclude missing target values from the corresponding "
        "horizon-specific training or evaluation dataset"
    ),
    "feature_history_policy": (
        "Historical predictor observations may cross the development "
        "and holdout origin-date boundary when available at forecast origin"
    ),
    "holdout_policy": (
        "Do not use final holdout data for feature selection, "
        "hyperparameter tuning, or model selection"
    ),
    "random_split_allowed": False,
    "holdout_used_for_training": False,
    "validation_period": None,
    "validation_period_note": (
        "A separate chronological validation period within development "
        "must be defined before model selection"
    )
}

split_policy_path = REPORT_DIR / "WI_03_chronological_split_policy.json"

with open(split_policy_path, "w", encoding="utf-8") as f:
    json.dump(split_policy, f, indent=2)

with open(split_policy_path, "r", encoding="utf-8") as f:
    verified_policy = json.load(f)

assert verified_policy["split_method"] == "chronological_by_target_date"
assert verified_policy["final_holdout"]["start"] == "2023-01-01"
assert verified_policy["holdout_used_for_training"] is False

print("SPLIT POLICY: SAVED AND VERIFIED")
print("Development: 2013–2022")
print("Final holdout: 2023–2024")
print("Horizons:", len(FORECAST_HORIZONS))
print("Assignment: target date")
print("Random split:", verified_policy["random_split_allowed"])
print("Holdout used for training:", verified_policy["holdout_used_for_training"])
print("Validation period: not yet defined")
print("Saved:", split_policy_path)


SPLIT POLICY: SAVED AND VERIFIED
Development: 2013–2022
Final holdout: 2023–2024
Horizons: 7
Assignment: target date
Random split: False
Holdout used for training: False
Validation period: not yet defined
Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_03_chronological_split_policy.json


In [58]:

# ============================================================
# WI-03 — CELL 43
# Define chronological train/validation/holdout periods
# ============================================================

import json

TRAIN_START = "2013-01-01"
TRAIN_END = "2019-12-31"

VALIDATION_START = "2020-01-01"
VALIDATION_END = "2022-12-31"

HOLDOUT_START = "2023-01-01"
HOLDOUT_END = "2024-12-31"

validation_policy = {
    "split_method": "chronological_by_target_date",
    "training": {
        "start": TRAIN_START,
        "end": TRAIN_END,
        "purpose": "Model fitting"
    },
    "validation": {
        "start": VALIDATION_START,
        "end": VALIDATION_END,
        "purpose": "Model selection and hyperparameter tuning"
    },
    "final_holdout": {
        "start": HOLDOUT_START,
        "end": HOLDOUT_END,
        "purpose": "One-time final evaluation"
    },
    "forecast_horizons": FORECAST_HORIZONS,
    "assignment_rule": (
        "For each horizon, assign rows using target_date = "
        "forecast_origin_date + horizon_days"
    ),
    "random_split_allowed": False,
    "holdout_used_for_training": False,
    "holdout_used_for_tuning": False
}

validation_policy_path = (
    REPORT_DIR / "WI_03_train_validation_holdout_policy.json"
)

assert not validation_policy_path.exists(), (
    "Policy file already exists. Inspect it before replacing."
)

with open(validation_policy_path, "w", encoding="utf-8") as f:
    json.dump(validation_policy, f, indent=2)

with open(validation_policy_path, "r", encoding="utf-8") as f:
    check = json.load(f)

assert check["training"]["start"] == "2013-01-01"
assert check["validation"]["end"] == "2022-12-31"
assert check["final_holdout"]["start"] == "2023-01-01"
assert check["holdout_used_for_training"] is False

print("THREE-WAY SPLIT POLICY: SAVED AND VERIFIED")
print("Training: 2013–2019")
print("Validation: 2020–2022")
print("Final holdout: 2023–2024")
print("Assignment: target date, horizon-specific")
print("Random split:", check["random_split_allowed"])
print("Holdout used for tuning:", check["holdout_used_for_tuning"])
print("Saved:", validation_policy_path)


THREE-WAY SPLIT POLICY: SAVED AND VERIFIED
Training: 2013–2019
Validation: 2020–2022
Final holdout: 2023–2024
Assignment: target date, horizon-specific
Random split: False
Holdout used for tuning: False
Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_03_train_validation_holdout_policy.json


In [59]:

# ============================================================
# WI-03 — CELL 44
# Audit actual train / validation / holdout membership
# ============================================================

import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

FEATURE_DATASET_DIR = FEATURE_OUTPUT_DIR / "weather-features-v1"

parquet_files = sorted(
    FEATURE_DATASET_DIR.rglob("batch_*.parquet")
)

assert len(parquet_files) == 252, (
    f"Expected 252 Parquet batches, found {len(parquet_files)}"
)

split_ranges = {
    "training": (
        np.datetime64(TRAIN_START, "D"),
        np.datetime64(TRAIN_END, "D")
    ),
    "validation": (
        np.datetime64(VALIDATION_START, "D"),
        np.datetime64(VALIDATION_END, "D")
    ),
    "final_holdout": (
        np.datetime64(HOLDOUT_START, "D"),
        np.datetime64(HOLDOUT_END, "D")
    )
}

audit = {}

for target in TARGET_VARIABLES:
    for horizon in FORECAST_HORIZONS:
        target_col = f"{target}_target_t_plus_{horizon}"
        audit_key = f"{target}_T_plus_{horizon}"

        audit[audit_key] = {
            split: {
                "rows": 0,
                "valid_targets": 0,
                "missing_targets": 0
            }
            for split in split_ranges
        }

for file_path in parquet_files:
    available_columns = ["date"] + [
        f"{target}_target_t_plus_{h}"
        for target in TARGET_VARIABLES
        for h in FORECAST_HORIZONS
    ]

    table = pq.read_table(
        file_path,
        columns=available_columns
    )
    frame = table.to_pandas()

    origin_dates = pd.to_datetime(
        frame["date"]
    ).to_numpy(dtype="datetime64[D]")

    for target in TARGET_VARIABLES:
        for horizon in FORECAST_HORIZONS:
            target_col = f"{target}_target_t_plus_{horizon}"
            target_dates = origin_dates + np.timedelta64(
                horizon, "D"
            )

            target_values = frame[target_col].to_numpy()
            valid_values = pd.notna(target_values)

            audit_key = f"{target}_T_plus_{horizon}"

            for split, (start_date, end_date) in split_ranges.items():
                mask = (
                    (target_dates >= start_date)
                    & (target_dates <= end_date)
                )

                row_count = int(mask.sum())
                valid_count = int(
                    (mask & valid_values).sum()
                )

                audit[audit_key][split]["rows"] += row_count
                audit[audit_key][split]["valid_targets"] += valid_count

                audit[audit_key][split]["missing_targets"] += (
                    row_count - valid_count
                )

    del frame, table

# ------------------------------------------------------------
# Assertions and summary
# ------------------------------------------------------------

summary_rows = []

for audit_key, split_results in audit.items():
    for split, counts in split_results.items():
        assert counts["rows"] > 0, (
            f"No rows found: {audit_key}, {split}"
        )

        assert counts["missing_targets"] == 0, (
            f"Missing targets found: {audit_key}, {split}: "
            f"{counts['missing_targets']}"
        )

        assert (
            counts["valid_targets"] == counts["rows"]
        )

        summary_rows.append({
            "target_horizon": audit_key,
            "split": split,
            "rows": counts["rows"],
            "valid_targets": counts["valid_targets"],
            "missing_targets": counts["missing_targets"]
        })

summary_df = pd.DataFrame(summary_rows)

# Check that each target/horizon has the same row count
# across the two target variables.
for horizon in FORECAST_HORIZONS:
    rainfall_rows = audit[
        f"PRECTOTCORR_T_plus_{horizon}"
    ]
    temperature_rows = audit[
        f"T2M_MAX_T_plus_{horizon}"
    ]

    assert rainfall_rows == temperature_rows, (
        f"Target row count mismatch at T+{horizon}"
    )

audit_path = REPORT_DIR / "WI_03_three_way_split_audit.json"

audit_output = {
    "dataset_version": "weather-features-v1",
    "split_method": "chronological_by_target_date",
    "split_ranges": {
        split: {
            "start": str(dates[0]),
            "end": str(dates[1])
        }
        for split, dates in split_ranges.items()
    },
    "batch_files_audited": len(parquet_files),
    "target_horizons_audited": len(audit),
    "results": audit,
    "all_split_targets_complete": True
}

with open(audit_path, "w", encoding="utf-8") as f:
    json.dump(audit_output, f, indent=2)

print("THREE-WAY TARGET-DATE SPLIT AUDIT: PASSED")
print("Parquet batches audited:", len(parquet_files))
print("Target/horizon combinations:", len(audit))
print()
print(
    summary_df.to_string(
        index=False,
        formatters={"rows": "{:,}".format,
                    "valid_targets": "{:,}".format,
                    "missing_targets": "{:,}".format}
    )
)
print()
print("Missing targets across assigned splits:",
      int(summary_df["missing_targets"].sum()))
print("Saved:", audit_path)


THREE-WAY TARGET-DATE SPLIT AUDIT: PASSED
Parquet batches audited: 252
Target/horizon combinations: 14

      target_horizon         split      rows valid_targets missing_targets
PRECTOTCORR_T_plus_1      training 6,438,600     6,438,600               0
PRECTOTCORR_T_plus_1    validation 2,761,920     2,761,920               0
PRECTOTCORR_T_plus_1 final_holdout 1,842,120     1,842,120               0
PRECTOTCORR_T_plus_2      training 6,436,080     6,436,080               0
PRECTOTCORR_T_plus_2    validation 2,761,920     2,761,920               0
PRECTOTCORR_T_plus_2 final_holdout 1,842,120     1,842,120               0
PRECTOTCORR_T_plus_3      training 6,433,560     6,433,560               0
PRECTOTCORR_T_plus_3    validation 2,761,920     2,761,920               0
PRECTOTCORR_T_plus_3 final_holdout 1,842,120     1,842,120               0
PRECTOTCORR_T_plus_4      training 6,431,040     6,431,040               0
PRECTOTCORR_T_plus_4    validation 2,761,920     2,761,920             

In [61]:

# ============================================================
# WI-03 — CELL 45A
# Diagnose non-finite predictor values
# ============================================================

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

feature_columns = FEATURE_COLUMNS

nonfinite_by_column = {
    col: {
        "count": 0,
        "positive_inf": 0,
        "negative_inf": 0
    }
    for col in feature_columns
}

examples = []
affected_cells = set()
affected_dates = set()
files_with_nonfinite = []

parquet_files = sorted(
    FEATURE_DATASET_DIR.rglob("batch_*.parquet")
)

for file_path in parquet_files:
    frame = pq.read_table(
        file_path,
        columns=["cell_id", "date"] + feature_columns
    ).to_pandas()

    values = frame[feature_columns].to_numpy(
        dtype=np.float64
    )

    nonfinite_mask = ~np.isfinite(values)

    if nonfinite_mask.any():
        files_with_nonfinite.append(str(file_path))

        rows, cols = np.where(nonfinite_mask)

        for col_idx in np.unique(cols):
            col = feature_columns[col_idx]
            col_values = values[:, col_idx]

            inf_pos = np.isposinf(col_values)
            inf_neg = np.isneginf(col_values)

            nonfinite_by_column[col]["positive_inf"] += int(
                inf_pos.sum()
            )
            nonfinite_by_column[col]["negative_inf"] += int(
                inf_neg.sum()
            )
            nonfinite_by_column[col]["count"] += int(
                (~np.isfinite(col_values)).sum()
            )

        for row_idx, col_idx in zip(rows[:100], cols[:100]):
            affected_cells.add(
                str(frame.iloc[row_idx]["cell_id"])
            )
            affected_dates.add(
                str(pd.Timestamp(frame.iloc[row_idx]["date"]).date())
            )

            if len(examples) < 30:
                examples.append({
                    "cell_id": str(frame.iloc[row_idx]["cell_id"]),
                    "date": str(
                        pd.Timestamp(frame.iloc[row_idx]["date"]).date()
                    ),
                    "feature": feature_columns[col_idx],
                    "value": repr(values[row_idx, col_idx])
                })

    del frame, values, nonfinite_mask

nonfinite_summary = pd.DataFrame([
    {
        "feature": col,
        **counts
    }
    for col, counts in nonfinite_by_column.items()
    if counts["count"] > 0
]).sort_values("count", ascending=False)

print("NON-FINITE PREDICTOR DIAGNOSTIC")
print("Files scanned:", len(parquet_files))
print("Files with non-finite values:", len(files_with_nonfinite))
print("Affected cells:", len(affected_cells))
print("Affected dates:", len(affected_dates))
print()

if nonfinite_summary.empty:
    print("No non-finite values found in this scan.")
else:
    print(nonfinite_summary.to_string(index=False))
    print()
    print("Example locations:")
    print(pd.DataFrame(examples).to_string(index=False))


NON-FINITE PREDICTOR DIAGNOSTIC
Files scanned: 252
Files with non-finite values: 252
Affected cells: 252
Affected dates: 3

                    feature  count  positive_inf  negative_inf
         PRECTOTCORR_lag_30  75600             0             0
             T2M_MAX_lag_30  75600             0             0
             T2M_MIN_lag_30  75600             0             0
     T2M_MAX_rolling_std_30  73080             0             0
    T2M_MAX_rolling_mean_30  73080             0             0
    T2M_MIN_rolling_mean_30  73080             0             0
PRECTOTCORR_rolling_mean_30  73080             0             0
 PRECTOTCORR_rolling_std_30  73080             0             0
     T2M_MIN_rolling_std_30  73080             0             0
         PRECTOTCORR_lag_14  35280             0             0
             T2M_MIN_lag_14  35280             0             0
             T2M_MAX_lag_14  35280             0             0
    T2M_MIN_rolling_mean_14  32760             0         

In [62]:

# ============================================================
# WI-03 — CELL 45B
# Corrected feature completeness and model-ready sample audit
# ============================================================

import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

feature_columns = FEATURE_COLUMNS

split_ranges = {
    "training": (
        np.datetime64(TRAIN_START, "D"),
        np.datetime64(TRAIN_END, "D")
    ),
    "validation": (
        np.datetime64(VALIDATION_START, "D"),
        np.datetime64(VALIDATION_END, "D")
    ),
    "final_holdout": (
        np.datetime64(HOLDOUT_START, "D"),
        np.datetime64(HOLDOUT_END, "D")
    )
}

feature_audit = {
    split: {
        "rows": 0,
        "complete_feature_rows": 0,
        "incomplete_feature_rows": 0,
        "missing_feature_values": 0,
        "positive_infinity_values": 0,
        "negative_infinity_values": 0
    }
    for split in split_ranges
}

model_ready_audit = {
    f"{target}_T_plus_{h}": {
        split: {
            "rows": 0,
            "model_ready_rows": 0,
            "incomplete_feature_rows": 0,
            "missing_target_rows": 0
        }
        for split in split_ranges
    }
    for target in TARGET_VARIABLES
    for h in FORECAST_HORIZONS
}

parquet_files = sorted(
    FEATURE_DATASET_DIR.rglob("batch_*.parquet")
)

assert len(parquet_files) == 252

for file_path in parquet_files:
    required_columns = (
        ["cell_id", "date"]
        + feature_columns
        + TARGET_COLUMNS
    )

    frame = pq.read_table(
        file_path,
        columns=required_columns
    ).to_pandas()

    origin_dates = pd.to_datetime(
        frame["date"]
    ).to_numpy(dtype="datetime64[D]")

    feature_frame = frame[feature_columns]

    feature_missing = feature_frame.isna().to_numpy()

    feature_values = feature_frame.to_numpy(
        dtype=np.float64
    )

    positive_inf = np.isposinf(feature_values)
    negative_inf = np.isneginf(feature_values)

    # NaN is allowed as structural missingness.
    # Infinity is never allowed.
    assert not positive_inf.any(), (
        f"Positive infinity found in {file_path}"
    )
    assert not negative_inf.any(), (
        f"Negative infinity found in {file_path}"
    )

    row_has_missing = feature_missing.any(axis=1)
    complete_features = ~row_has_missing

    for split, (start_date, end_date) in split_ranges.items():

        # Feature completeness is based on forecast-origin rows.
        # Count it once per origin, using T+1.
        target_dates_t1 = (
            origin_dates + np.timedelta64(1, "D")
        )

        split_mask_t1 = (
            (target_dates_t1 >= start_date)
            & (target_dates_t1 <= end_date)
        )

        positions_t1 = np.flatnonzero(split_mask_t1)

        feature_audit[split]["rows"] += len(positions_t1)
        feature_audit[split]["complete_feature_rows"] += int(
            complete_features[positions_t1].sum()
        )
        feature_audit[split]["incomplete_feature_rows"] += int(
            row_has_missing[positions_t1].sum()
        )
        feature_audit[split]["missing_feature_values"] += int(
            feature_missing[positions_t1].sum()
        )

        for target in TARGET_VARIABLES:
            for horizon in FORECAST_HORIZONS:

                target_dates = (
                    origin_dates + np.timedelta64(horizon, "D")
                )

                split_mask = (
                    (target_dates >= start_date)
                    & (target_dates <= end_date)
                )

                positions = np.flatnonzero(split_mask)

                target_col = (
                    f"{target}_target_t_plus_{horizon}"
                )

                target_values = frame[target_col].to_numpy()
                target_valid = pd.notna(target_values[positions])

                ready = (
                    complete_features[positions]
                    & target_valid
                )

                key = f"{target}_T_plus_{horizon}"
                result = model_ready_audit[key][split]

                result["rows"] += len(positions)
                result["model_ready_rows"] += int(ready.sum())
                result["incomplete_feature_rows"] += int(
                    row_has_missing[positions].sum()
                )
                result["missing_target_rows"] += int(
                    (~target_valid).sum()
                )

    del frame, feature_frame, feature_values

# ------------------------------------------------------------
# Validate counts
# ------------------------------------------------------------

for split, counts in feature_audit.items():
    assert (
        counts["complete_feature_rows"]
        + counts["incomplete_feature_rows"]
        == counts["rows"]
    )

for key, split_results in model_ready_audit.items():
    for split, counts in split_results.items():
        assert counts["missing_target_rows"] == 0
        assert (
            counts["model_ready_rows"]
            + counts["incomplete_feature_rows"]
            == counts["rows"]
        )

feature_summary = pd.DataFrame([
    {"split": split, **counts}
    for split, counts in feature_audit.items()
])

ready_summary = pd.DataFrame([
    {
        "target_horizon": key,
        "split": split,
        **counts
    }
    for key, split_results in model_ready_audit.items()
    for split, counts in split_results.items()
])

audit_output = {
    "dataset_version": "weather-features-v1",
    "feature_count": len(feature_columns),
    "target_count": len(TARGET_COLUMNS),
    "missing_values_policy": (
        "Structural NaNs retained; complete-feature rows required for "
        "model fitting and evaluation"
    ),
    "infinity_values_policy": "Positive and negative infinity prohibited",
    "feature_completeness_by_split": feature_audit,
    "model_ready_samples_by_target_horizon": model_ready_audit,
    "infinity_values_found": False
}

audit_path = (
    REPORT_DIR / "WI_03_feature_completeness_by_split.json"
)

assert not audit_path.exists(), (
    "Audit report already exists. Inspect before replacing."
)

with open(audit_path, "w", encoding="utf-8") as f:
    json.dump(audit_output, f, indent=2)

print("CORRECTED FEATURE COMPLETENESS AUDIT: PASSED")
print()
print("Feature completeness by split:")
print(feature_summary.to_string(index=False))
print()
print("Model-ready samples:")
print(
    ready_summary[
        ["target_horizon", "split", "model_ready_rows"]
    ].to_string(index=False)
)
print()
print("Positive/negative infinity values: 0")
print("Structural NaNs retained; no imputation performed.")
print("Saved:", audit_path)


CORRECTED FEATURE COMPLETENESS AUDIT: PASSED

Feature completeness by split:
        split    rows  complete_feature_rows  incomplete_feature_rows  missing_feature_values  positive_infinity_values  negative_infinity_values
     training 6438600                6363000                    75600                 1186920                         0                         0
   validation 2761920                2761920                        0                       0                         0                         0
final_holdout 1842120                1842120                        0                       0                         0                         0

Model-ready samples:
      target_horizon         split  model_ready_rows
PRECTOTCORR_T_plus_1      training           6363000
PRECTOTCORR_T_plus_1    validation           2761920
PRECTOTCORR_T_plus_1 final_holdout           1842120
PRECTOTCORR_T_plus_2      training           6360480
PRECTOTCORR_T_plus_2    validation           2761920


# WI-03 — Leakage-safe Feature Engineering: Completed

## 1. Objective

The objective of WI-03 was to transform the canonical `weather-daily-v1` dataset into a leakage-safe, multi-horizon weather forecasting feature dataset.

The feature engineering process was designed around a clearly defined forecast origin: the end of day *t*, after the weather observations for day *t* are available.

The resulting dataset supports daily precipitation and maximum-temperature forecasting for horizons T+1 through T+7 at the weather grid-cell level.

No model training was performed in this notebook.

## 2. Forecasting Targets

Two weather variables were selected as forecasting targets:

- `PRECTOTCORR` — daily precipitation (mm/day).
- `T2M_MAX` — daily maximum temperature (°C).

`T2M_MIN` was retained as an input predictor and was not configured as a forecasting target.

The forecasting horizons are:

- T+1
- T+2
- T+3
- T+4
- T+5
- T+6
- T+7

## 3. Feature Engineering

A total of 53 predictor features were generated.

| Feature group | Number of features |
|---|---:|
| Spatial features | 2 |
| Current-day weather features | 3 |
| Lag features | 18 |
| Rolling statistics | 24 |
| Calendar features | 6 |
| **Total predictors** | **53** |

### Temporal features

Lag periods:

- 1 day
- 2 days
- 3 days
- 7 days
- 14 days
- 30 days

Rolling windows:

- 3 days
- 7 days
- 14 days
- 30 days

Rolling mean and standard deviation were calculated for each weather variable.

Calendar features include day of year, month, day of week, and selected cyclical encodings.

Spatial predictors include latitude and longitude.

### Forecast-origin convention

All predictors follow the same availability rule:

- Current features represent observations on day *t*.
- `lag_1` represents day *t-1*.
- `lag_k` represents day *t-k*.
- Rolling windows end on day *t* and include the observation on day *t*.
- Target columns represent observations on day *t+h*.

No future weather observations are used as predictors.

## 4. Final Feature Dataset

The final dataset contains:

| Attribute | Result |
|---|---:|
| Grid cells | 2,520 |
| Observations | 11,045,160 |
| Predictor columns | 53 |
| Target columns | 14 |
| Identifier columns | 2 |
| Total columns | 69 |
| Parquet batches | 252 |
| Parquet storage | Approximately 2,088.66 MiB |

The dataset was saved under:

`phase3b_weather_intelligence/data/processed/weather_features_v1/weather-features-v1`

All 252 batches were registered in the batch manifest.

The final batch audit verified:

- No missing batch files.
- No row-count mismatches.
- No schema inconsistencies.
- No duplicate `(cell_id, date)` keys.
- No grid-cell coverage gaps or overlaps.
- Continuous daily coverage for every grid cell.

## 5. Missingness and Data Quality

Missing values were retained where they naturally occur because of the temporal structure of the dataset.

### Feature missingness

Initial missing feature values are caused by insufficient historical observations at the beginning of each grid-cell time series.

For example:

- `lag_1` requires one preceding day.
- `lag_7` requires seven preceding days.
- `lag_30` requires thirty preceding days.
- Rolling windows require their complete configured history, including the forecast-origin observation.

The structural missingness audit confirmed that these missing values occur at the beginning of each grid-cell series.

### Target missingness

Target values are unavailable at the end of each time series because the required future observations fall outside the archive.

For example, T+7 targets are unavailable for the final seven forecast-origin dates.

### Final quality audit

- Training contains 75,600 rows with incomplete predictors.
- Validation contains no incomplete predictor rows.
- Final holdout contains no incomplete predictor rows.
- No positive or negative infinity values were detected.
- No structural missing values were imputed or replaced with zero.

Incomplete predictor rows are excluded from model-ready samples. Target missingness is handled using horizon-specific masks.

## 6. Chronological Data Splitting

A three-way chronological split was established using target dates.

| Split | Target-date period | Purpose |
|---|---|---|
| Training | 2013–2019 | Model fitting |
| Validation | 2020–2022 | Model selection and hyperparameter tuning |
| Final holdout | 2023–2024 | Final evaluation |

Each forecast row is assigned independently for each horizon using:

`target_date = forecast_origin_date + horizon_days`

Random splitting is prohibited.

The final holdout is not used for training, feature selection, or hyperparameter tuning.

Historical predictor observations may cross split boundaries when they were available at the corresponding forecast origin.

## 7. Model-ready Samples

After applying the complete-feature requirement and horizon-specific target-date assignment, the model-ready sample counts were established.

### Training

- T+1: 6,363,000 rows.
- T+2: 6,360,480 rows.
- T+3: 6,357,960 rows.
- T+4: 6,355,440 rows.
- T+5: 6,352,920 rows.
- T+6: 6,350,400 rows.
- T+7: 6,347,880 rows.

These counts apply independently to both forecasting targets.

### Validation

2,761,920 model-ready rows for each target and horizon.

### Final holdout

1,842,120 model-ready rows for each target and horizon.

All target values within the assigned split periods were verified as available.

## 8. Leakage Prevention

The following controls were implemented:

1. Forecast-origin availability was explicitly defined.
2. Lag features use only historical observations.
3. Rolling windows end at the forecast origin.
4. Target dates are calculated independently for every horizon.
5. Chronological splitting is based on target dates.
6. Random splitting is prohibited.
7. Final holdout data is excluded from training and tuning.
8. Climatological features were not calculated using the complete archive.
9. APY, crop-yield, and future weather information were not introduced as predictors.

## 9. Limitations

- The dataset represents weather grid cells, not validated administrative districts.
- Spatial interpretation must not be treated as district-level intelligence.
- Structural missing values remain in the canonical feature dataset.
- The archive ends in 2024 and cannot independently provide operational live weather forecasts in 2026.
- Physical accuracy of extreme weather observations requires independent validation.
- No predictive model performance has been established in this notebook.

## 10. Artifacts and Reports

Primary dataset:

- `weather-features-v1`

Important reports:

- `WI_03_feature_specification.json`
- `WI_03_chronological_split_policy.json`
- `WI_03_train_validation_holdout_policy.json`
- `WI_03_three_way_split_audit.json`
- `WI_03_feature_quality_audit.json`
- `WI_03_feature_completeness_by_split.json`
- `batch_manifest.json`

## 11. Conclusion

WI-03 successfully completed the leakage-safe feature engineering pipeline for the Weather Intelligence 2.0 forecasting system.

The resulting feature dataset contains 53 predictors, 14 forecasting targets, and seven forecast horizons across 2,520 weather grid cells.

The complete dataset was generated, stored in 252 Parquet batches, and audited for schema consistency, key integrity, spatial and temporal coverage, structural missingness, target availability, and chronological split correctness.

Training, validation, and final holdout samples have been defined without using random splitting or imputing structural missing values.

**WI-03 is complete. No model training or performance claims are made at this stage.**

**Next notebook: WI-04 — Baseline Weather Forecasting.**
